In [ ]:
# In[1]:

import pandas as pd
import pytz
from collections import OrderedDict

tz = pytz.timezone('Asia/Shanghai')

files = ["metric.csv", "trace.csv", "log.csv", "error_logs.csv"]

# map file -> name_field (None for error_logs)
name_field_map = {
    "metric.csv": "kpi_name",
    "trace.csv": "trace_name",
    "log.csv": "log_name",
    "error_logs.csv": None
}

summary = OrderedDict()

for fn in files:
    try:
        df = pd.read_csv(fn)
    except Exception as e:
        # If file missing or unreadable, store an error entry
        summary[fn] = {
            "error": str(e)
        }
        continue

    # total rows
    total_rows = int(df.shape[0])

    # timestamps: compute min/max if timestamp column exists
    min_ts_str = None
    max_ts_str = None
    if "timestamp" in df.columns and not df["timestamp"].dropna().empty:
        # convert once and reuse
        ts_series = pd.to_datetime(df["timestamp"].astype(int), unit="s", utc=True).dt.tz_convert(tz)
        min_ts = ts_series.min()
        max_ts = ts_series.max()
        # represent as ISO strings (compact)
        min_ts_str = pd.Timestamp(min_ts).isoformat()
        max_ts_str = pd.Timestamp(max_ts).isoformat()
    else:
        ts_series = None

    # unique cmdb_id values (alphabetically sorted, top 20)
    if "cmdb_id" in df.columns:
        unique_cmdb = sorted(df["cmdb_id"].dropna().astype(str).unique())
        unique_cmdb_top = unique_cmdb[:20]
    else:
        unique_cmdb = []
        unique_cmdb_top = []

    # unique names for kpi/trace/log where applicable
    name_field = name_field_map.get(fn)
    unique_names = None
    unique_names_limited = None
    if name_field and name_field in df.columns:
        unique_names = sorted(df[name_field].dropna().astype(str).unique())
        if len(unique_names) > 50:
            unique_names_limited = unique_names[:50]
        else:
            unique_names_limited = unique_names

    # sample: first 10 rows; make a shallow copy and convert timestamp for readability if present
    sample = df.head(10).copy()
    if ts_series is not None:
        # attach a human-readable timestamp column for sample
        sample["_datetime_Asia_Shanghai"] = ts_series.head(10).astype(str).values

    # build compact summary for this file
    summary[fn] = {
        "total_rows": total_rows,
        "min_timestamp_Asia_Shanghai": min_ts_str,
        "max_timestamp_Asia_Shanghai": max_ts_str,
        "unique_cmdb_id_count": len(unique_cmdb),
        "unique_cmdb_id_top20": unique_cmdb_top,
        "name_field": name_field,
        "unique_name_values_count": len(unique_names) if unique_names is not None else None,
        "unique_name_values_list_limited": unique_names_limited,
        "sample_first_10_rows": sample  # DataFrame (compact)
    }

# Display the compact structured summary dict
summary

```
Out[1]:
```
```python
# Summarize the previously computed 'summary' OrderedDict in plain English.
# Reuse the in-memory 'summary' variable from the prior execution.
metric_s = summary["metric.csv"]
trace_s = summary["trace.csv"]
log_s = summary["log.csv"]
err_s = summary["error_logs.csv"]

result = (
    "Telemetry files summary (all times in Asia/Shanghai, UTC+8):\n\n"
    "metric.csv:\n"
    f"- Total rows: {metric_s['total_rows']}\n"
    f"- Time range: {metric_s['min_timestamp_Asia_Shanghai']} to {metric_s['max_timestamp_Asia_Shanghai']}\n"
    f"- Unique cmdb_id count: {metric_s['unique_cmdb_id_count']}; top 20 (alphabetical): {', '.join(metric_s['unique_cmdb_id_top20'])}\n"
    f"- Unique kpi_name count: {metric_s['unique_name_values_count']} (list limited to 50 shown in earlier output)\n"
    "- Sample: first 10 rows were included in the earlier output.\n\n"
    "trace.csv:\n"
    f"- Total rows: {trace_s['total_rows']}\n"
    f"- Time range: {trace_s['min_timestamp_Asia_Shanghai']} to {trace_s['max_timestamp_Asia_Shanghai']}\n"
    f"- Unique cmdb_id count: {trace_s['unique_cmdb_id_count']}; top 20 (alphabetical): {', '.join(trace_s['unique_cmdb_id_top20'])}\n"
    f"- Unique trace_name count: {trace_s['unique_name_values_count']} (list limited to 50 shown in earlier output)\n"
    "- Sample: first 10 rows were included in the earlier output.\n\n"
    "log.csv:\n"
    f"- Total rows: {log_s['total_rows']}\n"
    f"- Time range: {log_s['min_timestamp_Asia_Shanghai']} to {log_s['max_timestamp_Asia_Shanghai']}\n"
    f"- Unique cmdb_id count: {log_s['unique_cmdb_id_count']}; top 20 (alphabetical): {', '.join(log_s['unique_cmdb_id_top20'])}\n"
    f"- Unique log_name values ({log_s['unique_name_values_count']}): {', '.join(log_s['unique_name_values_list_limited'])}\n"
    "- Sample: first 10 rows were included in the earlier output.\n\n"
    "error_logs.csv:\n"
    f"- Total rows: {err_s['total_rows']}\n"
    f"- Time range: {err_s['min_timestamp_Asia_Shanghai']} to {err_s['max_timestamp_Asia_Shanghai']}\n"
    f"- Unique cmdb_id count: {err_s['unique_cmdb_id_count']}; values: {', '.join(err_s['unique_cmdb_id_top20'])}\n"
    "- Contains textual messages (sample of first 10 rows shown earlier); these appear to be warnings from frontend pods.\n\n"
    "Overall: All four files loaded successfully. metric.csv is the largest (144k+ rows) with many KPIs, trace.csv has ~37k rows, log.csv is small and has two log metrics, and error_logs.csv contains ~2k warning messages from frontend pods."
)

result
```

The original code execution output of IPython Kernel is also provided below for reference:

OrderedDict([('metric.csv', {'total_rows': 144252, 'min_timestamp_Asia_Shanghai': '2022-03-20T12:30:00+08:00', 'max_timestamp_Asia_Shanghai': '2022-03-20T12:59:00+08:00', 'unique_cmdb_id_count': 59, 'unique_cmdb_id_top20': ['adservice', 'adservice-0', 'adservice-1', 'adservice-2', 'adservice2', 'adservice2-0', 'cartservice', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'cartservice2-0', 'checkoutservice', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'checkoutservice2-0', 'currencyservice', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2'], 'name_field': 'kpi_name', 'unique_name_values_count': 1152, 'unique_name_values_list_limited': ['app.grpc.count', 'app.grpc.mrt', 'app.grpc.rr', 'app.grpc.sr', 'app.http.count', 'app.http.mrt', 'app.http.rr', 'app.http.sr', 'container.node-5.container_cpu_cfs_periods', 'container.node-5.container_cpu_cfs_throttled_periods', 'container.node-5.container_cpu_cfs_throttled_seconds', 'container.node-5.container_cpu_load_average_10s', 'container.node-5.container_cpu_system_seconds', 'container.node-5.container_cpu_usage_seconds', 'container.node-5.container_cpu_user_seconds', 'container.node-5.container_file_descriptors', 'container.node-5.container_fs_inodes./dev/vda1', 'container.node-5.container_fs_inodes_free./dev/vda1', 'container.node-5.container_fs_io_current./dev/vda1', 'container.node-5.container_fs_io_time_seconds./dev/vda1', 'container.node-5.container_fs_io_time_weighted_seconds./dev/vda1', 'container.node-5.container_fs_limit_MB./dev/vda1', 'container.node-5.container_fs_read_seconds./dev/vda1', 'container.node-5.container_fs_reads./dev/vda', 'container.node-5.container_fs_reads./dev/vda1', 'container.node-5.container_fs_reads_MB./dev/vda', 'container.node-5.container_fs_reads_merged./dev/vda1', 'container.node-5.container_fs_sector_reads./dev/vda1', 'container.node-5.container_fs_sector_writes./dev/vda1', 'container.node-5.container_fs_usage_MB./dev/vda1', 'container.node-5.container_fs_write_seconds./dev/vda1', 'container.node-5.container_fs_writes./dev/vda', 'container.node-5.container_fs_writes./dev/vda1', 'container.node-5.container_fs_writes_MB./dev/vda', 'container.node-5.container_fs_writes_merged./dev/vda1', 'container.node-5.container_last_seen', 'container.node-5.container_memory_cache', 'container.node-5.container_memory_failcnt', 'container.node-5.container_memory_failures.container.pgfault', 'container.node-5.container_memory_failures.container.pgmajfault', 'container.node-5.container_memory_failures.hierarchy.pgfault', 'container.node-5.container_memory_failures.hierarchy.pgmajfault', 'container.node-5.container_memory_mapped_file', 'container.node-5.container_memory_max_usage_MB', 'container.node-5.container_memory_rss', 'container.node-5.container_memory_swap', 'container.node-5.container_memory_usage_MB', 'container.node-5.container_memory_working_set_MB', 'container.node-5.container_network_receive_MB.eth0', 'container.node-5.container_network_receive_errors.eth0'], 'sample_first_10_rows':     timestamp    cmdb_id                                           kpi_name        value    _datetime_Asia_Shanghai
0  1647750600  adservice                                     app.grpc.count   232.000000  2022-03-20 12:30:00+08:00
1  1647750600  adservice                                       app.grpc.mrt     0.002269  2022-03-20 12:30:00+08:00
2  1647750600  adservice                                        app.grpc.rr   100.000000  2022-03-20 12:30:00+08:00
3  1647750600  adservice                                        app.grpc.sr    55.603448  2022-03-20 12:30:00+08:00
4  1647750600  adservice                                     app.http.count    24.000000  2022-03-20 12:30:00+08:00
5  1647750600  adservice                                       app.http.mrt     0.008810  2022-03-20 12:30:00+08:00
6  1647750600  adservice                                        app.http.rr   100.000000  2022-03-20 12:30:00+08:00
7  1647750600  adservice                                        app.http.sr   100.000000  2022-03-20 12:30:00+08:00
8  1647750600  adservice    runtime.java_lang_ClassLoading_LoadedClassCount  4990.000000  2022-03-20 12:30:00+08:00
9  1647750600  adservice  runtime.java_lang_ClassLoading_TotalLoadedClas...  4990.000000  2022-03-20 12:30:00+08:00}), ('trace.csv', {'total_rows': 36656, 'min_timestamp_Asia_Shanghai': '2022-03-20T12:30:00+08:00', 'max_timestamp_Asia_Shanghai': '2022-03-20T12:59:00+08:00', 'unique_cmdb_id_count': 40, 'unique_cmdb_id_top20': ['adservice-0', 'adservice-1', 'adservice-2', 'adservice2-0', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'cartservice2-0', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'checkoutservice2-0', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'currencyservice2-0', 'emailservice-0', 'emailservice-1', 'emailservice-2', 'emailservice2-0'], 'name_field': 'trace_name', 'unique_name_values_count': 200, 'unique_name_values_list_limited': ['trace.from_checkoutservice-0.duration_mean', 'trace.from_checkoutservice-0.duration_p95', 'trace.from_checkoutservice-0.error_rate', 'trace.from_checkoutservice-0.row_count', 'trace.from_checkoutservice-1.duration_mean', 'trace.from_checkoutservice-1.duration_p95', 'trace.from_checkoutservice-1.error_rate', 'trace.from_checkoutservice-1.row_count', 'trace.from_checkoutservice-2.duration_mean', 'trace.from_checkoutservice-2.duration_p95', 'trace.from_checkoutservice-2.error_rate', 'trace.from_checkoutservice-2.row_count', 'trace.from_checkoutservice2-0.duration_mean', 'trace.from_checkoutservice2-0.duration_p95', 'trace.from_checkoutservice2-0.error_rate', 'trace.from_checkoutservice2-0.row_count', 'trace.from_frontend-0.duration_mean', 'trace.from_frontend-0.duration_p95', 'trace.from_frontend-0.error_rate', 'trace.from_frontend-0.row_count', 'trace.from_frontend-1.duration_mean', 'trace.from_frontend-1.duration_p95', 'trace.from_frontend-1.error_rate', 'trace.from_frontend-1.row_count', 'trace.from_frontend-2.duration_mean', 'trace.from_frontend-2.duration_p95', 'trace.from_frontend-2.error_rate', 'trace.from_frontend-2.row_count', 'trace.from_frontend2-0.duration_mean', 'trace.from_frontend2-0.duration_p95', 'trace.from_frontend2-0.error_rate', 'trace.from_frontend2-0.row_count', 'trace.from_recommendationservice-0.duration_mean', 'trace.from_recommendationservice-0.duration_p95', 'trace.from_recommendationservice-0.error_rate', 'trace.from_recommendationservice-0.row_count', 'trace.from_recommendationservice-1.duration_mean', 'trace.from_recommendationservice-1.duration_p95', 'trace.from_recommendationservice-1.error_rate', 'trace.from_recommendationservice-1.row_count', 'trace.from_recommendationservice-2.duration_mean', 'trace.from_recommendationservice-2.duration_p95', 'trace.from_recommendationservice-2.error_rate', 'trace.from_recommendationservice-2.row_count', 'trace.from_recommendationservice2-0.duration_mean', 'trace.from_recommendationservice2-0.duration_p95', 'trace.from_recommendationservice2-0.error_rate', 'trace.from_recommendationservice2-0.row_count', 'trace.from_root.duration_mean', 'trace.from_root.duration_p95'], 'sample_first_10_rows':     timestamp      cmdb_id                           trace_name      value    _datetime_Asia_Shanghai
0  1647750600  adservice-0  trace.from_frontend-0.duration_mean   0.000012  2022-03-20 12:30:00+08:00
1  1647750600  adservice-0   trace.from_frontend-0.duration_p95   0.000018  2022-03-20 12:30:00+08:00
2  1647750600  adservice-0     trace.from_frontend-0.error_rate   0.000000  2022-03-20 12:30:00+08:00
3  1647750600  adservice-0      trace.from_frontend-0.row_count  22.000000  2022-03-20 12:30:00+08:00
4  1647750600  adservice-0  trace.from_frontend-1.duration_mean   0.000012  2022-03-20 12:30:00+08:00
5  1647750600  adservice-0   trace.from_frontend-1.duration_p95   0.000013  2022-03-20 12:30:00+08:00
6  1647750600  adservice-0     trace.from_frontend-1.error_rate   0.000000  2022-03-20 12:30:00+08:00
7  1647750600  adservice-0      trace.from_frontend-1.row_count  21.000000  2022-03-20 12:30:00+08:00
8  1647750600  adservice-0  trace.from_frontend-2.duration_mean   0.000013  2022-03-20 12:30:00+08:00
9  1647750600  adservice-0   trace.from_frontend-2.duration_p95   0.000020  2022-03-20 12:30:00+08:00}), ('log.csv', {'total_rows': 1814, 'min_timestamp_Asia_Shanghai': '2022-03-20T12:30:00+08:00', 'max_timestamp_Asia_Shanghai': '2022-03-20T12:59:00+08:00', 'unique_cmdb_id_count': 31, 'unique_cmdb_id_top20': ['adservice-0', 'adservice-1', 'adservice-2', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'emailservice-0', 'emailservice-1', 'emailservice-2', 'frontend-0', 'frontend-1', 'frontend-2', 'paymentservice-0', 'paymentservice-1'], 'name_field': 'log_name', 'unique_name_values_count': 2, 'unique_name_values_list_limited': ['log.error_count', 'log.row_count'], 'sample_first_10_rows':     timestamp        cmdb_id         log_name   value    _datetime_Asia_Shanghai
0  1647750600    adservice-0  log.error_count     0.0  2022-03-20 12:30:00+08:00
1  1647750600    adservice-0    log.row_count  1528.0  2022-03-20 12:30:00+08:00
2  1647750600    adservice-1  log.error_count     0.0  2022-03-20 12:30:00+08:00
3  1647750600    adservice-1    log.row_count   136.0  2022-03-20 12:30:00+08:00
4  1647750600    adservice-2  log.error_count     0.0  2022-03-20 12:30:00+08:00
5  1647750600    adservice-2    log.row_count   138.0  2022-03-20 12:30:00+08:00
6  1647750600  cartservice-0  log.error_count     0.0  2022-03-20 12:30:00+08:00
7  1647750600  cartservice-0    log.row_count  1024.0  2022-03-20 12:30:00+08:00
8  1647750600  cartservice-1  log.error_count     0.0  2022-03-20 12:30:00+08:00
9  1647750600  cartservice-1    log.row_count  1068.0  2022-03-20 12:30:00+08:00}), ('error_logs.csv', {'total_rows': 1994, 'min_timestamp_Asia_Shanghai': '2022-03-20T12:30:00+08:00', 'max_timestamp_Asia_Shanghai': '2022-03-20T12:59:59+08:00', 'unique_cmdb_id_count': 3, 'unique_cmdb_id_top20': ['frontend-0', 'frontend-1', 'frontend-2'], 'name_field': None, 'unique_name_values_count': None, 'unique_name_values_list_limited': None, 'sample_first_10_rows':     timestamp     cmdb_id                                            message    _datetime_Asia_Shanghai
0  1647750600  frontend-1  severity: warning, message: failed to retrieve...  2022-03-20 12:30:00+08:00
1  1647750600  frontend-2  severity: warning, message: failed to retrieve...  2022-03-20 12:30:00+08:00
2  1647750603  frontend-0  severity: warning, message: failed to retrieve...  2022-03-20 12:30:03+08:00
3  1647750604  frontend-1  severity: warning, message: failed to retrieve...  2022-03-20 12:30:04+08:00
4  1647750605  frontend-2  severity: warning, message: failed to retrieve...  2022-03-20 12:30:05+08:00
5  1647750605  frontend-0  severity: warning, message: failed to retrieve...  2022-03-20 12:30:05+08:00
6  1647750606  frontend-1  severity: warning, message: failed to retrieve...  2022-03-20 12:30:06+08:00
7  1647750607  frontend-2  severity: warning, message: failed to retrieve...  2022-03-20 12:30:07+08:00
8  1647750608  frontend-2  severity: warning, message: failed to retrieve...  2022-03-20 12:30:08+08:00
9  1647750608  frontend-0  severity: warning, message: failed to retrieve...  2022-03-20 12:30:08+08:00})])```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load metric.csv (reuse variable if exists in kernel)
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

# Ensure correct dtypes
metric_df['timestamp'] = metric_df['timestamp'].astype(int)
metric_df['cmdb_id'] = metric_df['cmdb_id'].astype(str)
metric_df['kpi_name'] = metric_df['kpi_name'].astype(str)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# 1) Compute global P95 and P05 per (cmdb_id, kpi_name) using full series (before any filtering)
thresholds_df = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], observed=True)['value']
    .agg(global_P95=lambda x: x.quantile(0.95), global_P05=lambda x: x.quantile(0.05))
    .reset_index()
)

# 2) Filter metric.csv rows to the incident window (inclusive) in Asia/Shanghai timezone
start_ts = int(pd.Timestamp("2022-03-20T12:30:00+08:00", tz=tz).tz_convert('UTC').timestamp())
end_ts = int(pd.Timestamp("2022-03-20T12:59:59+08:00", tz=tz).tz_convert('UTC').timestamp())

window_df = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

# 3) Mark anomalies within the filtered window using the precomputed global thresholds
# Merge thresholds into windowed data
window_df = window_df.merge(thresholds_df, on=['cmdb_id', 'kpi_name'], how='left')

# Anomaly condition: value >= P95 OR value <= P05
window_df['anomaly'] = (window_df['value'] >= window_df['global_P95']) | (window_df['value'] <= window_df['global_P05'])

# 4A) Top 20 (cmdb_id,kpi_name) combos ranked by anomaly count
# Compute anomaly aggregates
anomaly_groups = (
    window_df[window_df['anomaly']]
    .groupby(['cmdb_id', 'kpi_name'], observed=True)
    .agg(
        anomaly_count=('anomaly', 'sum'),
        earliest_anomaly_timestamp=('timestamp', 'min')
    )
    .reset_index()
)

# Compute max value in window per combo (regardless of anomaly)
max_in_window = (
    window_df
    .groupby(['cmdb_id', 'kpi_name'], observed=True)['value']
    .agg(max_value_in_window='max')
    .reset_index()
)

# Combine aggregates with thresholds
top20_df = (
    anomaly_groups
    .merge(max_in_window, on=['cmdb_id', 'kpi_name'], how='left')
    .merge(thresholds_df, on=['cmdb_id', 'kpi_name'], how='left')
)

# Format earliest timestamp to Asia/Shanghai ISO string for readability
top20_df['earliest_anomaly_timestamp'] = pd.to_datetime(top20_df['earliest_anomaly_timestamp'], unit='s', utc=True).dt.tz_convert(tz).astype(str)

# Order and select top 20 by anomaly_count desc
top20_df = top20_df.sort_values(by='anomaly_count', ascending=False).head(20)[
    ['cmdb_id', 'kpi_name', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window', 'global_P95', 'global_P05']
].reset_index(drop=True)

# 4B) Top 10 cmdb_id aggregated by total anomaly_count
comp_agg = (
    window_df[window_df['anomaly']]
    .groupby('cmdb_id', observed=True)
    .agg(
        total_anomaly_count=('anomaly', 'sum'),
        earliest_anomaly_timestamp_over_all_KPIs=('timestamp', 'min')
    )
    .reset_index()
)

# Format timestamp and select top 10 components
comp_agg['earliest_anomaly_timestamp_over_all_KPIs'] = pd.to_datetime(comp_agg['earliest_anomaly_timestamp_over_all_KPIs'], unit='s', utc=True).dt.tz_convert(tz).astype(str)
top10_comp_df = comp_agg.sort_values(by='total_anomaly_count', ascending=False).head(10).reset_index(drop=True)

# Keep results compact (top20_df and top10_comp_df)
top20_df, top10_comp_df

```
Out[2]:
```
```python
# Summarize the anomaly analysis results in plain English using cached top20_df and top10_comp_df.
# Reuse existing variables from prior execution (top20_df, top10_comp_df).
try:
    top20_df
    top10_comp_df
except NameError:
    raise RuntimeError("Expected variables top20_df and top10_comp_df to exist in the kernel.")

# Build a compact plain-English summary
top20_sample = top20_df.head(10)[['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp']].to_dict(orient='records')
top_components = top10_comp_df.to_dict(orient='records')

summary_lines = [
    "Anomaly detection summary (window: 2022-03-20 12:30:00+08 to 2022-03-20 12:59:59+08):",
    "",
    "Method:",
    "- Global P95 and P05 were computed per (cmdb_id, kpi_name) using the entire metric.csv series.",
    "- Rows inside the incident window were marked anomalous when value >= global_P95 OR value <= global_P05.",
    "",
    "Top findings (compact):",
    f"- Total distinct (cmdb_id,kpi_name) combos with highest anomaly counts (showing top 10 of top 20):"
]

# add the top20 sample lines
for r in top20_sample:
    summary_lines.append(f"  * {r['cmdb_id']} | {r['kpi_name']} — anomalies: {int(r['anomaly_count'])}, earliest: {r['earliest_anomaly_timestamp']}")

summary_lines += [
    "",
    "- Top components by aggregated anomaly count (top 10):"
]

# add the top components lines
for c in top_components:
    summary_lines.append(f"  * {c['cmdb_id']} — total_anomalies: {int(c['total_anomaly_count'])}, earliest_anomaly: {c['earliest_anomaly_timestamp_over_all_KPIs']}")

summary_lines += [
    "",
    "Interpretation / notes:",
    "- Many of the highest-anomaly (cmdb_id,kpi_name) rows in the top20 are from shippingservice2-0 (multiple container.fs and container metrics), each with anomaly_count = 30 and earliest anomaly at 2022-03-20 12:30:00+08. Several of these metrics have max_value equal to their global thresholds (suggesting constant/edge values).",
    "- The component-level aggregates show adservice and adservice2 as the largest anomaly sources (adservice: 9206 anomalies; adservice2: 8999 anomalies). Several frontend pods and productcatalogservice pods also appear high on the list.",
    "- Earliest anomalies across top combos/components are all at the window start (2022-03-20 12:30:00+08), indicating anomalies present for the entire 30-minute window for many KPIs.",
    "",
    "If you want the full top-20 combo table or the underlying thresholds for further inspection, I can return those compactly as well."
]

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id                                           kpi_name  anomaly_count earliest_anomaly_timestamp  max_value_in_window    global_P95    global_P05
0   shippingservice2-0  container.node-5.container_ulimits_soft.max_op...             30  2022-03-20 12:30:00+08:00         6.553600e+04  6.553600e+04  6.553600e+04
1   shippingservice2-0      container.node-5.container_fs_writes./dev/vda             30  2022-03-20 12:30:00+08:00         3.300000e+01  1.500000e+00  0.000000e+00
2   shippingservice2-0     container.node-5.container_fs_writes./dev/vda1             30  2022-03-20 12:30:00+08:00         0.000000e+00  0.000000e+00  0.000000e+00
3   shippingservice2-0   container.node-5.container_fs_writes_MB./dev/vda             30  2022-03-20 12:30:00+08:00         2.304688e-01  1.367188e-02  0.000000e+00
4   shippingservice2-0  container.node-5.container_fs_writes_merged./d...             30  2022-03-20 12:30:00+08:00         0.000000e+00  0.000000e+00  0.000000e+00
5   shippingservice2-0      container.node-5.container_fs_reads./dev/vda1             30  2022-03-20 12:30:00+08:00         0.000000e+00  0.000000e+00  0.000000e+00
6   shippingservice2-0  container.node-5.container_fs_reads_merged./de...             30  2022-03-20 12:30:00+08:00         0.000000e+00  0.000000e+00  0.000000e+00
7   shippingservice2-0  container.node-5.container_fs_sector_reads./de...             30  2022-03-20 12:30:00+08:00         0.000000e+00  0.000000e+00  0.000000e+00
8   shippingservice2-0  container.node-5.container_fs_sector_writes./d...             30  2022-03-20 12:30:00+08:00         0.000000e+00  0.000000e+00  0.000000e+00
9   shippingservice2-0  container.node-5.container_fs_write_seconds./d...             30  2022-03-20 12:30:00+08:00         0.000000e+00  0.000000e+00  0.000000e+00
10  shippingservice2-0     container.node-5.container_fs_inodes./dev/vda1             30  2022-03-20 12:30:00+08:00         2.000000e+01  0.000000e+00  0.000000e+00
11  shippingservice2-0  container.node-5.container_fs_inodes_free./dev...             30  2022-03-20 12:30:00+08:00         0.000000e+00  0.000000e+00  0.000000e+00
12  shippingservice2-0  container.node-5.container_fs_io_current./dev/...             30  2022-03-20 12:30:00+08:00         0.000000e+00  0.000000e+00  0.000000e+00
13  shippingservice2-0  container.node-5.container_fs_io_time_seconds....             30  2022-03-20 12:30:00+08:00         0.000000e+00  0.000000e+00  0.000000e+00
14  shippingservice2-0  container.node-5.container_fs_io_time_weighted...             30  2022-03-20 12:30:00+08:00         0.000000e+00  0.000000e+00  0.000000e+00
15  shippingservice2-0   container.node-5.container_fs_limit_MB./dev/vda1             30  2022-03-20 12:30:00+08:00         6.046307e+05  6.046307e+05  6.046307e+05
16  shippingservice2-0  container.node-5.container_fs_read_seconds./de...             30  2022-03-20 12:30:00+08:00         0.000000e+00  0.000000e+00  0.000000e+00
17  shippingservice2-0    container.node-5.container_cpu_load_average_10s             30  2022-03-20 12:30:00+08:00         0.000000e+00  0.000000e+00  0.000000e+00
18           adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...             30  2022-03-20 12:30:00+08:00        -1.000000e+00 -1.000000e+00 -1.000000e+00
19           adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...             30  2022-03-20 12:30:00+08:00         5.242880e+06  5.242880e+06  5.242880e+06,                    cmdb_id  total_anomaly_count earliest_anomaly_timestamp_over_all_KPIs
0                adservice                 9206                2022-03-20 12:30:00+08:00
1               adservice2                 8999                2022-03-20 12:30:00+08:00
2              frontend2-0                 3010                2022-03-20 12:30:00+08:00
3               frontend-0                 2937                2022-03-20 12:30:00+08:00
4               frontend-1                 2850                2022-03-20 12:30:00+08:00
5               frontend-2                 2508                2022-03-20 12:30:00+08:00
6        checkoutservice-0                 2350                2022-03-20 12:30:00+08:00
7  productcatalogservice-2                 2341                2022-03-20 12:30:00+08:00
8  productcatalogservice-0                 2319                2022-03-20 12:30:00+08:00
9  productcatalogservice-1                 2219                2022-03-20 12:30:00+08:00)```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse error_logs_df if available, else load
try:
    error_logs_df
except NameError:
    error_logs_df = pd.read_csv("error_logs.csv")

# Ensure types
error_logs_df['timestamp'] = error_logs_df['timestamp'].astype(int)
error_logs_df['cmdb_id'] = error_logs_df['cmdb_id'].astype(str)
if 'message' in error_logs_df.columns:
    error_logs_df['message'] = error_logs_df['message'].astype(str)
else:
    error_logs_df['message'] = ""

# Incident window (inclusive) in Asia/Shanghai -> convert to UTC epoch seconds
start_ts = int(pd.Timestamp("2022-03-20T12:30:00+08:00", tz=tz).tz_convert('UTC').timestamp())
end_ts = int(pd.Timestamp("2022-03-20T12:59:59+08:00", tz=tz).tz_convert('UTC').timestamp())

# 1) Filter rows to the incident window
err_window_df = error_logs_df[(error_logs_df['timestamp'] >= start_ts) & (error_logs_df['timestamp'] <= end_ts)].copy()

# Attach Asia/Shanghai datetime and minute floor string for grouping
err_window_df['_dt_shanghai'] = pd.to_datetime(err_window_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
err_window_df['_minute_shanghai'] = err_window_df['_dt_shanghai'].dt.floor('min').astype(str)

# 2) Aggregate counts per minute and list top 10 minutes by count
top10_minutes_df = (
    err_window_df
    .groupby('_minute_shanghai', observed=True)
    .size()
    .reset_index(name='total_count')
    .sort_values(by='total_count', ascending=False)
    .head(10)
    .reset_index(drop=True)
)

# 3) For the top 2 minutes, get per-cmdb_id counts (top 10) and up to 10 sample messages
top_minutes = top10_minutes_df['_minute_shanghai'].head(2).tolist()

per_minute_cmdb_counts = {}
per_minute_samples = {}
for minute in top_minutes:
    df_min = err_window_df[err_window_df['_minute_shanghai'] == minute]
    # a) per-cmdb_id counts (top 10)
    per_cmdb = (
        df_min
        .groupby('cmdb_id', observed=True)
        .size()
        .reset_index(name='count')
        .sort_values(by='count', ascending=False)
        .head(10)
        .reset_index(drop=True)
    )
    per_minute_cmdb_counts[minute] = per_cmdb
    # b) up to 10 sample messages (timestamp, cmdb_id, message) - include readable datetime
    samples = df_min[['timestamp', 'cmdb_id', 'message', '_dt_shanghai']].sort_values(by='timestamp').head(10).reset_index(drop=True)
    per_minute_samples[minute] = samples

# 4) Overall distinct cmdb_ids present in the window and their total counts (top 20)
top20_cmdb_counts_df = (
    err_window_df
    .groupby('cmdb_id', observed=True)
    .size()
    .reset_index(name='total_count')
    .sort_values(by='total_count', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

# Compact outputs
top10_minutes_df, per_minute_cmdb_counts, per_minute_samples, top20_cmdb_counts_df

```
Out[3]:
```
```python
# Summarize error_logs analysis in plain English using cached variables:
# top10_minutes_df, per_minute_cmdb_counts, per_minute_samples, top20_cmdb_counts_df

try:
    top10_minutes_df
    per_minute_cmdb_counts
    per_minute_samples
    top20_cmdb_counts_df
except NameError:
    raise RuntimeError("Expected analysis variables are not present in the kernel.")

# Build concise plain-English summary
m_list = top10_minutes_df.to_dict(orient='records')
minute_lines = [f"{r['_minute_shanghai']}: {int(r['total_count'])}" for r in m_list]

# Top 2 minutes details
top2 = top10_minutes_df['_minute_shanghai'].head(2).tolist()
details_lines = []
for minute in top2:
    per_cmdb = per_minute_cmdb_counts.get(minute)
    samples = per_minute_samples.get(minute)
    # top cmdb_id counts (up to 10)
    cmdb_lines = []
    if per_cmdb is not None and not per_cmdb.empty:
        for _, row in per_cmdb.iterrows():
            cmdb_lines.append(f"{row['cmdb_id']} ({int(row['count'])})")
    # sample messages (up to 10)
    sample_lines = []
    if samples is not None and not samples.empty:
        for _, s in samples.iterrows():
            ts = s['_dt_shanghai']
            sample_lines.append(f"{ts} {s['cmdb_id']}: {s['message']}")
    details_lines.append({
        "minute": minute,
        "top_cmdb_ids": ", ".join(cmdb_lines) if cmdb_lines else "none",
        "sample_messages": sample_lines
    })

# Top cmdb_ids overall in window (top 20 limited earlier)
top_cmdb_overall = top20_cmdb_counts_df.to_dict(orient='records')

summary = [
    "Error logs summary (window: 2022-03-20 12:30:00+08 → 2022-03-20 12:59:59+08):",
    "",
    "1) Top 10 minutes by error-log count (minute: total_count):",
] + [f"  - {line}" for line in minute_lines] + [
    "",
    "2) Details for top 2 minutes:",
]

for d in details_lines:
    summary += [
        f"  - Minute: {d['minute']}",
        f"    Top cmdb_ids (count): {d['top_cmdb_ids']}",
        f"    Sample messages (up to 10):"
    ]
    if d['sample_messages']:
        for s in d['sample_messages']:
            summary.append(f"      * {s}")
    else:
        summary.append("      * (no samples)")

summary += [
    "",
    "3) Overall distinct cmdb_ids in the window (top entries, cmdb_id: total_count):"
]

for r in top_cmdb_overall:
    summary.append(f"  - {r['cmdb_id']}: {int(r['total_count'])}")

summary += [
    "",
    "Interpretation:",
    "- Error logs are concentrated on frontend pods. Peak minutes are 2022-03-20 12:49 and 12:45 (each 73 logs).",
    "- During those peak minutes the top sources are frontend-2, frontend-0, and frontend-1.",
    "- Sample messages are warnings like 'failed to retrieve...'.",
    "- Overall counts (window): frontend-2 (763), frontend-0 (695), frontend-1 (536).",
]

result = "\n".join(summary)
result
```

The original code execution output of IPython Kernel is also provided below for reference:

(            _minute_shanghai  total_count
0  2022-03-20 12:49:00+08:00           73
1  2022-03-20 12:45:00+08:00           73
2  2022-03-20 12:38:00+08:00           72
3  2022-03-20 12:47:00+08:00           72
4  2022-03-20 12:32:00+08:00           71
5  2022-03-20 12:43:00+08:00           70
6  2022-03-20 12:33:00+08:00           70
7  2022-03-20 12:30:00+08:00           69
8  2022-03-20 12:41:00+08:00           68
9  2022-03-20 12:35:00+08:00           68, {'2022-03-20 12:49:00+08:00':       cmdb_id  count
0  frontend-2     28
1  frontend-0     26
2  frontend-1     19, '2022-03-20 12:45:00+08:00':       cmdb_id  count
0  frontend-2     29
1  frontend-0     25
2  frontend-1     19}, {'2022-03-20 12:49:00+08:00':     timestamp     cmdb_id                                            message              _dt_shanghai
0  1647751740  frontend-1  severity: warning, message: failed to retrieve... 2022-03-20 12:49:00+08:00
1  1647751741  frontend-2  severity: warning, message: failed to retrieve... 2022-03-20 12:49:01+08:00
2  1647751742  frontend-2  severity: warning, message: failed to retrieve... 2022-03-20 12:49:02+08:00
3  1647751742  frontend-0  severity: warning, message: failed to retrieve... 2022-03-20 12:49:02+08:00
4  1647751743  frontend-2  severity: warning, message: failed to retrieve... 2022-03-20 12:49:03+08:00
5  1647751744  frontend-0  severity: warning, message: failed to retrieve... 2022-03-20 12:49:04+08:00
6  1647751745  frontend-0  severity: warning, message: failed to retrieve... 2022-03-20 12:49:05+08:00
7  1647751746  frontend-0  severity: warning, message: failed to retrieve... 2022-03-20 12:49:06+08:00
8  1647751746  frontend-2  severity: warning, message: failed to retrieve... 2022-03-20 12:49:06+08:00
9  1647751748  frontend-1  severity: warning, message: failed to retrieve... 2022-03-20 12:49:08+08:00, '2022-03-20 12:45:00+08:00':     timestamp     cmdb_id                                            message              _dt_shanghai
0  1647751500  frontend-0  severity: warning, message: failed to retrieve... 2022-03-20 12:45:00+08:00
1  1647751503  frontend-2  severity: warning, message: failed to retrieve... 2022-03-20 12:45:03+08:00
2  1647751503  frontend-0  severity: warning, message: failed to retrieve... 2022-03-20 12:45:03+08:00
3  1647751504  frontend-0  severity: warning, message: failed to retrieve... 2022-03-20 12:45:04+08:00
4  1647751505  frontend-1  severity: warning, message: failed to retrieve... 2022-03-20 12:45:05+08:00
5  1647751505  frontend-2  severity: warning, message: failed to retrieve... 2022-03-20 12:45:05+08:00
6  1647751506  frontend-2  severity: warning, message: failed to retrieve... 2022-03-20 12:45:06+08:00
7  1647751508  frontend-0  severity: warning, message: failed to retrieve... 2022-03-20 12:45:08+08:00
8  1647751509  frontend-2  severity: warning, message: failed to retrieve... 2022-03-20 12:45:09+08:00
9  1647751510  frontend-0  severity: warning, message: failed to retrieve... 2022-03-20 12:45:10+08:00},       cmdb_id  total_count
0  frontend-2          763
1  frontend-0          695
2  frontend-1          536)```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse trace_df if present, else load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

# Ensure correct dtypes
trace_df['timestamp'] = trace_df['timestamp'].astype(int)
trace_df['cmdb_id'] = trace_df['cmdb_id'].astype(str)
trace_df['trace_name'] = trace_df['trace_name'].astype(str)
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# 1) Compute global P95 and P05 per (cmdb_id, trace_name) using the full series (before filtering)
thresholds_df = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], observed=True)['value']
    .agg(global_P95=lambda x: x.quantile(0.95), global_P05=lambda x: x.quantile(0.05))
    .reset_index()
)

# 2) Filter trace.csv to the incident window (inclusive) in Asia/Shanghai (converted to UTC epoch seconds)
start_ts = int(pd.Timestamp("2022-03-20T12:30:00+08:00", tz=tz).tz_convert('UTC').timestamp())
end_ts = int(pd.Timestamp("2022-03-20T12:59:59+08:00", tz=tz).tz_convert('UTC').timestamp())

window_df = trace_df[(trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] <= end_ts)].copy()

# 3) Merge thresholds and mark anomalies using global thresholds
window_df = window_df.merge(thresholds_df, on=['cmdb_id', 'trace_name'], how='left')
window_df['anomaly'] = (window_df['value'] >= window_df['global_P95']) | (window_df['value'] <= window_df['global_P05'])

# 4A) Top 20 (cmdb_id, trace_name) combos ranked by anomaly count
anomaly_groups = (
    window_df[window_df['anomaly']]
    .groupby(['cmdb_id', 'trace_name'], observed=True)
    .agg(
        anomaly_count=('anomaly', 'sum'),
        earliest_anomaly_timestamp=('timestamp', 'min')
    )
    .reset_index()
)

# Compute max/min in window per combo
window_stats = (
    window_df
    .groupby(['cmdb_id', 'trace_name'], observed=True)['value']
    .agg(max_value_in_window='max', min_value_in_window='min')
    .reset_index()
)

top20_df = (
    anomaly_groups
    .merge(window_stats, on=['cmdb_id', 'trace_name'], how='left')
    .merge(thresholds_df, on=['cmdb_id', 'trace_name'], how='left')
)

# Format earliest anomaly timestamp to Asia/Shanghai ISO string
top20_df['earliest_anomaly_timestamp'] = pd.to_datetime(top20_df['earliest_anomaly_timestamp'], unit='s', utc=True).dt.tz_convert(tz).astype(str)

top20_df = top20_df.sort_values(by='anomaly_count', ascending=False).head(20)[
    ['cmdb_id', 'trace_name', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window', 'min_value_in_window', 'global_P95', 'global_P05']
].reset_index(drop=True)

# 4B) For trace_name containing 'from_frontend' (any cmdb_id), top up to 20 combos by anomaly_count
frontend_mask = anomaly_groups['trace_name'].str.contains('from_frontend', na=False)
frontend_anoms = anomaly_groups[frontend_mask].copy()

frontend_top_df = (
    frontend_anoms
    .merge(window_stats, on=['cmdb_id', 'trace_name'], how='left')
    .merge(thresholds_df, on=['cmdb_id', 'trace_name'], how='left')
)

frontend_top_df['earliest_anomaly_timestamp'] = pd.to_datetime(frontend_top_df['earliest_anomaly_timestamp'], unit='s', utc=True).dt.tz_convert(tz).astype(str)

frontend_top_df = frontend_top_df.sort_values(by='anomaly_count', ascending=False).head(20)[
    ['cmdb_id', 'trace_name', 'anomaly_count', 'earliest_anomaly_timestamp', 'max_value_in_window', 'min_value_in_window', 'global_P95', 'global_P05']
].reset_index(drop=True)

# Return compact DataFrames (limited rows as requested)
top20_df, frontend_top_df

```
Out[4]:
```
```python
# Summarize the trace analysis results in plain English using cached top20_df and frontend_top_df.
try:
    top20_df
    frontend_top_df
except NameError:
    raise RuntimeError("Expected variables top20_df and frontend_top_df to exist in the kernel.")

summary = (
    "Trace anomaly summary (window: 2022-03-20 12:30:00+08 to 2022-03-20 12:59:59+08):\n\n"
    "Method:\n"
    "- Global P95 and P05 were computed per (cmdb_id, trace_name) using the entire trace.csv series.\n"
    "- Rows inside the incident window were flagged anomalous when value >= global_P95 OR value <= global_P05.\n\n"
    "Key findings (compact):\n"
    "- The top 20 (cmdb_id, trace_name) combos all show anomaly_count = 30 and earliest anomaly at 2022-03-20 12:30:00+08, indicating anomalies persist across the entire 30-minute window for these combos.\n"
    "- Many top anomalies are for trace.error_rate metrics (e.g., trace.from_frontend-*.error_rate) and have max and min values equal to 0.0 with global P95/P05 = 0.0, i.e., the series is at an edge value (zero) and considered anomalous by the percentile rule.\n"
    "- A few duration metrics (e.g., trace.from_frontend-0.duration_mean on adservice-0) show small non-zero values; for that combo global_P95 ≈ 0.000013 and global_P05 ≈ 0.000012 with max/min in-window spanning those ranges.\n"
    "- The affected components are widespread: adservice (multiple replicas), shippingservice and shippingservice2, recommendationservice, cartservice, and several others appear in the top lists.\n\n"
    "Specific subset (trace_name containing 'from_frontend'):\n"
    "- Up to 20 combos with 'from_frontend' were ranked; these also mostly have anomaly_count = 30 and earliest anomaly at the window start.\n"
    "- Most 'from_frontend' anomalies are error_rate metrics pinned at 0.0; one duration_mean (adservice-0) shows small duration values with P95/P05 around 1e-5.\n\n"
    "Interpretation / notes:\n"
    "- The anomaly rule (value vs global P95/P05) flags many zero-valued error_rate traces as anomalous because their global percentiles are also zero; this can indicate constant/edge-value telemetry rather than transient spikes.\n"
    "- The uniform earliest timestamps and counts suggest persistent, not isolated, deviations across the window for these trace metrics.\n\n"
    "If you want, I can (a) return the compact top-20 table or (b) show only those combos where max_value_in_window differs substantially from global percentiles for deeper investigation."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id                               trace_name  anomaly_count earliest_anomaly_timestamp  max_value_in_window  min_value_in_window  global_P95  global_P05
0          adservice-0         trace.from_frontend-2.error_rate             30  2022-03-20 12:30:00+08:00             0.000000             0.000000    0.000000    0.000000
1   shippingservice2-0        trace.from_frontend2-0.error_rate             30  2022-03-20 12:30:00+08:00             0.000000             0.000000    0.000000    0.000000
2          adservice-0      trace.from_frontend-0.duration_mean             30  2022-03-20 12:30:00+08:00             0.000084             0.000011    0.000013    0.000012
3          adservice-0         trace.from_frontend-1.error_rate             30  2022-03-20 12:30:00+08:00             0.000000             0.000000    0.000000    0.000000
4          adservice-0         trace.from_frontend-0.error_rate             30  2022-03-20 12:30:00+08:00             0.000000             0.000000    0.000000    0.000000
5    shippingservice-2         trace.from_frontend-2.error_rate             30  2022-03-20 12:30:00+08:00             0.000000             0.000000    0.000000    0.000000
6          adservice-1         trace.from_frontend-1.error_rate             30  2022-03-20 12:30:00+08:00             0.000000             0.000000    0.000000    0.000000
7    shippingservice-1  trace.from_checkoutservice-1.error_rate             30  2022-03-20 12:30:00+08:00             0.000000             0.000000    0.000000    0.000000
8    shippingservice-1  trace.from_checkoutservice-2.error_rate             30  2022-03-20 12:30:00+08:00             0.000000             0.000000    0.000000    0.000000
9    shippingservice-0         trace.from_frontend-2.error_rate             30  2022-03-20 12:30:00+08:00             0.000000             0.000000    0.000000    0.000000
10   shippingservice-1  trace.from_checkoutservice-0.error_rate             30  2022-03-20 12:30:00+08:00             0.000000             0.000000    0.000000    0.000000
11   shippingservice-0         trace.from_frontend-0.error_rate             30  2022-03-20 12:30:00+08:00             0.000000             0.000000    0.000000    0.000000
12   shippingservice-0         trace.from_frontend-1.error_rate             30  2022-03-20 12:30:00+08:00             0.000000             0.000000    0.000000    0.000000
13   shippingservice-0  trace.from_checkoutservice-1.error_rate             30  2022-03-20 12:30:00+08:00             0.000000             0.000000    0.000000    0.000000
14         adservice-1         trace.from_frontend-0.error_rate             30  2022-03-20 12:30:00+08:00             0.000000             0.000000    0.000000    0.000000
15   shippingservice-0  trace.from_checkoutservice-2.error_rate             30  2022-03-20 12:30:00+08:00             0.000000             0.000000    0.000000    0.000000
16         adservice-2         trace.from_frontend-0.error_rate             30  2022-03-20 12:30:00+08:00             0.000000             0.000000    0.000000    0.000000
17       cartservice-0         trace.from_frontend-2.error_rate             30  2022-03-20 12:30:00+08:00             0.000000             0.000000    0.000000    0.000000
18       cartservice-0                 trace.self.duration_mean             30  2022-03-20 12:30:00+08:00             0.000000             0.000000    0.000000    0.000000
19       cartservice-0                  trace.self.duration_p95             30  2022-03-20 12:30:00+08:00             0.000000             0.000000    0.000000    0.000000,                      cmdb_id                           trace_name  anomaly_count earliest_anomaly_timestamp  max_value_in_window  min_value_in_window  global_P95  global_P05
0                adservice-0  trace.from_frontend-0.duration_mean             30  2022-03-20 12:30:00+08:00             0.000084             0.000011    0.000013    0.000012
1                adservice-0     trace.from_frontend-0.error_rate             30  2022-03-20 12:30:00+08:00             0.000000             0.000000    0.000000    0.000000
2                adservice-0     trace.from_frontend-1.error_rate             30  2022-03-20 12:30:00+08:00             0.000000             0.000000    0.000000    0.000000
3         shippingservice2-0    trace.from_frontend2-0.error_rate             30  2022-03-20 12:30:00+08:00             0.000000             0.000000    0.000000    0.000000
4    recommendationservice-1     trace.from_frontend-2.error_rate             30  2022-03-20 12:30:00+08:00             0.000000             0.000000    0.000000    0.000000
5    recommendationservice-2     trace.from_frontend-2.error_rate             30  2022-03-20 12:30:00+08:00             0.000000             0.000000    0.000000    0.000000
6                adservice-1     trace.from_frontend-1.error_rate             30  2022-03-20 12:30:00+08:00             0.000000             0.000000    0.000000    0.000000
7   recommendationservice2-0    trace.from_frontend2-0.error_rate             30  2022-03-20 12:30:00+08:00             0.000000             0.000000    0.000000    0.000000
8                adservice-1     trace.from_frontend-0.error_rate             30  2022-03-20 12:30:00+08:00             0.000000             0.000000    0.000000    0.000000
9                adservice-0     trace.from_frontend-2.error_rate             30  2022-03-20 12:30:00+08:00             0.000000             0.000000    0.000000    0.000000
10   recommendationservice-1     trace.from_frontend-1.error_rate             30  2022-03-20 12:30:00+08:00             0.000000             0.000000    0.000000    0.000000
11   recommendationservice-2     trace.from_frontend-0.error_rate             30  2022-03-20 12:30:00+08:00             0.000000             0.000000    0.000000    0.000000
12   recommendationservice-1     trace.from_frontend-0.error_rate             30  2022-03-20 12:30:00+08:00             0.000000             0.000000    0.000000    0.000000
13         shippingservice-0     trace.from_frontend-0.error_rate             30  2022-03-20 12:30:00+08:00             0.000000             0.000000    0.000000    0.000000
14   recommendationservice-2     trace.from_frontend-1.error_rate             30  2022-03-20 12:30:00+08:00             0.000000             0.000000    0.000000    0.000000
15   recommendationservice-0     trace.from_frontend-1.error_rate             30  2022-03-20 12:30:00+08:00             0.000000             0.000000    0.000000    0.000000
16         checkoutservice-0     trace.from_frontend-2.error_rate             30  2022-03-20 12:30:00+08:00             0.000000             0.000000    0.000000    0.000000
17         currencyservice-0     trace.from_frontend-1.error_rate             30  2022-03-20 12:30:00+08:00             0.000000             0.000000    0.000000    0.000000
18         currencyservice-2     trace.from_frontend-0.error_rate             30  2022-03-20 12:30:00+08:00             0.000000             0.000000    0.000000    0.000000
19         currencyservice-1     trace.from_frontend-0.error_rate             30  2022-03-20 12:30:00+08:00             0.000000             0.000000    0.000000    0.000000)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz
from math import fabs

tz = pytz.timezone('Asia/Shanghai')

# Reuse metric_df if exists
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

# Selected cmdb_ids
selected = [
    "frontend-0", "frontend-1", "frontend-2",
    "adservice", "adservice2",
    "productcatalogservice-0", "productcatalogservice-1", "productcatalogservice-2",
    "shippingservice2-0"
]

# Ensure correct dtypes
metric_df['timestamp'] = metric_df['timestamp'].astype(int)
metric_df['cmdb_id'] = metric_df['cmdb_id'].astype(str)
metric_df['kpi_name'] = metric_df['kpi_name'].astype(str)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# 1) Compute global P95 and P05 per (cmdb_id, kpi_name) for selected cmdb_ids (using full series)
thresholds_sel = (
    metric_df[metric_df['cmdb_id'].isin(selected)]
    .groupby(['cmdb_id', 'kpi_name'], observed=True)['value']
    .agg(global_P95=lambda x: x.quantile(0.95), global_P05=lambda x: x.quantile(0.05))
    .reset_index()
)

# 2) Incident window (inclusive) -> compute epoch seconds for Asia/Shanghai
start_ts = int(pd.Timestamp("2022-03-20T12:30:00+08:00", tz=tz).tz_convert('UTC').timestamp())
end_ts   = int(pd.Timestamp("2022-03-20T12:59:59+08:00", tz=tz).tz_convert('UTC').timestamp())

window_sel = metric_df[
    (metric_df['timestamp'] >= start_ts) &
    (metric_df['timestamp'] <= end_ts) &
    (metric_df['cmdb_id'].isin(selected))
].copy()

# 3) Merge thresholds and mark anomalies
window_sel = window_sel.merge(thresholds_sel, on=['cmdb_id', 'kpi_name'], how='left')

# Fill NaN thresholds with 0 to allow comparisons; NaN values will not be anomalies if value is NaN
window_sel['global_P95'] = window_sel['global_P95'].fillna(0.0)
window_sel['global_P05'] = window_sel['global_P05'].fillna(0.0)

window_sel['anomaly'] = ((window_sel['value'] >= window_sel['global_P95']) | (window_sel['value'] <= window_sel['global_P05'])) & (~window_sel['value'].isna())

# 4) Generate consecutive-anomaly runs (consecutive-minute timestamps) per (cmdb_id, kpi_name)
anoms = window_sel[window_sel['anomaly']].sort_values(['cmdb_id', 'kpi_name', 'timestamp']).copy()

def extract_runs_for_group(g):
    # g is anomalies for one (cmdb_id,kpi_name)
    if g.empty:
        return pd.DataFrame([])
    g = g.sort_values('timestamp').copy()
    # mark new run when timestamp gap != 60 seconds
    is_new = (g['timestamp'].diff().fillna(9999) != 60)
    run_id = is_new.cumsum()
    g['run_id'] = run_id
    # aggregate per run
    runs = (
        g.groupby('run_id', observed=True)
         .agg(
             cmdb_id=('cmdb_id', 'first'),
             kpi_name=('kpi_name', 'first'),
             run_start_ts=('timestamp', 'min'),
             run_end_ts=('timestamp', 'max'),
             run_length_minutes=('timestamp', 'size'),
             max_value_in_run=('value', 'max'),
             min_value_in_run=('value', 'min'),
             global_P95=('global_P95', 'first'),
             global_P05=('global_P05', 'first')
         )
         .reset_index(drop=True)
    )
    # keep only runs length >= 3
    runs = runs[runs['run_length_minutes'] >= 3].copy()
    return runs

# apply per group
runs_list = []
grp = anoms.groupby(['cmdb_id', 'kpi_name'], observed=True)
for _, g in grp:
    r = extract_runs_for_group(g)
    if not r.empty:
        runs_list.append(r)

if runs_list:
    runs_df = pd.concat(runs_list, ignore_index=True)
else:
    runs_df = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','run_start_ts','run_end_ts','run_length_minutes',
        'max_value_in_run','min_value_in_run','global_P95','global_P05'
    ])

# 5) Compute severity score per run
if not runs_df.empty:
    def severity_row(row):
        gp95 = row['global_P95']
        gp05 = row['global_P05']
        mv = row['max_value_in_run']
        nv = row['min_value_in_run']
        # denominators per rule: if global is 0 use 1
        denom95 = abs(gp95) if gp95 != 0 else 1.0
        denom05 = abs(gp05) if gp05 != 0 else 1.0
        s1 = abs(mv - gp95) / denom95
        s2 = abs(gp05 - nv) / denom05
        return max(s1, s2)
    runs_df['severity'] = runs_df.apply(severity_row, axis=1)
    # convert start/end timestamps to Asia/Shanghai ISO strings
    runs_df['run_start_Asia_Shanghai'] = pd.to_datetime(runs_df['run_start_ts'], unit='s', utc=True).dt.tz_convert(tz).astype(str)
    runs_df['run_end_Asia_Shanghai'] = pd.to_datetime(runs_df['run_end_ts'], unit='s', utc=True).dt.tz_convert(tz).astype(str)
    # reorder columns and sort
    runs_df = runs_df[[
        'cmdb_id','kpi_name','run_start_Asia_Shanghai','run_end_Asia_Shanghai',
        'run_length_minutes','max_value_in_run','min_value_in_run','global_P95','global_P05','severity'
    ]]
    runs_top20 = runs_df.sort_values(by=['run_length_minutes','severity'], ascending=[False, False]).head(20).reset_index(drop=True)
else:
    runs_top20 = runs_df.copy()

# 6) Per-cmdb_id summary for selected set: total number of runs and earliest run_start (if any)
if not runs_df.empty:
    per_cmdb = (
        runs_df
        .groupby('cmdb_id', observed=True)
        .agg(
            total_runs=('kpi_name', 'size'),
            earliest_run_start_ts=('run_start_Asia_Shanghai', 'min')
        )
        .reset_index()
    )
else:
    per_cmdb = pd.DataFrame({'cmdb_id': selected})
    per_cmdb['total_runs'] = 0
    per_cmdb['earliest_run_start_ts'] = None

# Ensure all selected cmdb_ids present in summary
per_cmdb = per_cmdb.set_index('cmdb_id').reindex(selected).reset_index()
per_cmdb['total_runs'] = per_cmdb['total_runs'].fillna(0).astype(int)
per_cmdb = per_cmdb.rename(columns={'earliest_run_start_ts':'earliest_run_start'})

# Final compact outputs: runs_top20 (max 20 rows) and per_cmdb summary (one row per selected cmdb_id)
runs_top20, per_cmdb

```
Out[5]:
```
```python
# Summarize the focused fault-identification scan in plain English using cached results.
# Reuse runs_top20 and per_cmdb from prior execution and present a compact summary string.

try:
    runs_top20
    per_cmdb
except NameError:
    raise RuntimeError("Expected variables 'runs_top20' and 'per_cmdb' to exist in the kernel.")

# Prepare a few representative top runs (first 6) for concise reporting
top_runs_sample = runs_top20.head(6)[[
    'cmdb_id','kpi_name','run_length_minutes','severity','run_start_Asia_Shanghai','run_end_Asia_Shanghai',
    'max_value_in_run','min_value_in_run','global_P95','global_P05'
]].to_dict(orient='records')

# Prepare per-cmdb compact summary for the selected set
per_cmdb_records = per_cmdb[['cmdb_id','total_runs','earliest_run_start']].to_dict(orient='records')

lines = [
    "Focused fault-identification summary for selected components (window: 2022-03-20 12:30 → 12:59:59 +08):",
    "",
    "High-level findings:",
    "- Many anomaly runs span the entire 30-minute incident window (run_length = 30 minutes) for multiple (cmdb_id, kpi_name) combos.",
    "- The most severe and longest runs are on shippingservice2-0 (container fs/write/inode metrics) with very high severity scores (e.g., 20+), indicating large deviations from global percentiles.",
    "- adservice and adservice2 show many persistent runs (e.g., Java memory/finalizer and JVM/thread metrics) across the full window with notable counts.",
    "- productcatalogservice and frontend pods show persistent file-descriptor and socket-related runs (moderate severity).",
    "",
    "Representative top runs (showing up to 6 examples):"
]

for r in top_runs_sample:
    lines.append(
        f"  - {r['cmdb_id']} | {r['kpi_name']} — length: {int(r['run_length_minutes'])}m, "
        f"severity: {r['severity']:.3f}, start: {r['run_start_Asia_Shanghai']}, end: {r['run_end_Asia_Shanghai']}, "
        f"max: {r['max_value_in_run']}, min: {r['min_value_in_run']}, P95: {r['global_P95']}, P05: {r['global_P05']}"
    )

lines += [
    "",
    "Per-component summary (selected components):"
]

for c in per_cmdb_records:
    lines.append(f"  - {c['cmdb_id']}: total_runs = {int(c['total_runs'])}, earliest_run_start = {c['earliest_run_start']}")

lines += [
    "",
    "Interpretation / next steps:",
    "- shippingservice2-0 exhibits the most pronounced sustained deviations (filesystem/write/inode metrics) and should be prioritized for investigation.",
    "- adservice/adservice2 show many JVM-related persistent anomalies—investigate GC / memory-finalizer and thread state metrics.",
    "- Frontend and productcatalogservice pods have persistent resource-related anomalies (file descriptors, sockets) worth checking for resource exhaustion or leaks.",
    "- All earliest runs start at the window start (2022-03-20 12:30:00+08), indicating issues persisted throughout the incident window.",
    "",
    "If you want, I can (a) return the full compact top-20 runs table or (b) extract the raw rows for a specific run for deeper inspection (limited to samples)."
]

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id                                           kpi_name    run_start_Asia_Shanghai      run_end_Asia_Shanghai  run_length_minutes  max_value_in_run  min_value_in_run   global_P95  global_P05   severity
0        shippingservice2-0      container.node-5.container_fs_writes./dev/vda  2022-03-20 12:30:00+08:00  2022-03-20 12:59:00+08:00                  30         33.000000               0.0     1.500000         0.0  21.000000
1        shippingservice2-0     container.node-5.container_fs_inodes./dev/vda1  2022-03-20 12:30:00+08:00  2022-03-20 12:59:00+08:00                  30         20.000000               0.0     0.000000         0.0  20.000000
2        shippingservice2-0   container.node-5.container_fs_writes_MB./dev/vda  2022-03-20 12:30:00+08:00  2022-03-20 12:59:00+08:00                  30          0.230469               0.0     0.013672         0.0  15.857143
3                adservice2  runtime.java_lang_Memory_ObjectPendingFinaliza...  2022-03-20 12:30:00+08:00  2022-03-20 12:59:00+08:00                  30          0.500000               0.0     0.229167         0.0   1.181818
4                 adservice  runtime.java_lang_Memory_ObjectPendingFinaliza...  2022-03-20 12:30:00+08:00  2022-03-20 12:59:00+08:00                  30          0.416667               0.0     0.000000         0.0   0.416667
5   productcatalogservice-1                 container.node-5.container_sockets  2022-03-20 12:30:00+08:00  2022-03-20 12:59:00+08:00                  30          4.500000               4.0     4.000000         4.0   0.125000
6   productcatalogservice-2                 container.node-5.container_sockets  2022-03-20 12:30:00+08:00  2022-03-20 12:59:00+08:00                  30          4.500000               4.0     4.000000         4.0   0.125000
7                frontend-0                 container.node-5.container_sockets  2022-03-20 12:30:00+08:00  2022-03-20 12:59:00+08:00                  30         15.500000              14.0    14.275000        14.0   0.085814
8                frontend-0        container.node-5.container_file_descriptors  2022-03-20 12:30:00+08:00  2022-03-20 12:59:00+08:00                  30         21.500000              20.0    20.275000        20.0   0.060419
9   productcatalogservice-0                 container.node-5.container_sockets  2022-03-20 12:30:00+08:00  2022-03-20 12:59:00+08:00                  30          4.500000               4.0     4.275000         4.0   0.052632
10  productcatalogservice-1        container.node-5.container_file_descriptors  2022-03-20 12:30:00+08:00  2022-03-20 12:59:00+08:00                  30         10.500000              10.0    10.000000        10.0   0.050000
11  productcatalogservice-2        container.node-5.container_file_descriptors  2022-03-20 12:30:00+08:00  2022-03-20 12:59:00+08:00                  30         10.500000              10.0    10.000000        10.0   0.050000
12  productcatalogservice-0        container.node-5.container_file_descriptors  2022-03-20 12:30:00+08:00  2022-03-20 12:59:00+08:00                  30         10.500000              10.0    10.275000        10.0   0.021898
13                adservice                 runtime.jvm_threads_state.RUNNABLE  2022-03-20 12:30:00+08:00  2022-03-20 12:59:00+08:00                  30          5.083333               5.0     5.000000         5.0   0.016667
14               frontend-2  container.node-5.container_cpu_cfs_throttled_s...  2022-03-20 12:30:00+08:00  2022-03-20 12:59:00+08:00                  30          0.001502               0.0     0.000000         0.0   0.001502
15                adservice                                        app.grpc.rr  2022-03-20 12:30:00+08:00  2022-03-20 12:59:00+08:00                  30        100.000000             100.0   100.000000       100.0   0.000000
16                adservice                                     app.http.count  2022-03-20 12:30:00+08:00  2022-03-20 12:59:00+08:00                  30         24.000000              24.0    24.000000        24.0   0.000000
17                adservice                                        app.http.rr  2022-03-20 12:30:00+08:00  2022-03-20 12:59:00+08:00                  30        100.000000             100.0   100.000000       100.0   0.000000
18                adservice                                        app.http.sr  2022-03-20 12:30:00+08:00  2022-03-20 12:59:00+08:00                  30        100.000000             100.0   100.000000       100.0   0.000000
19                adservice    runtime.java_lang_ClassLoading_LoadedClassCount  2022-03-20 12:30:00+08:00  2022-03-20 12:59:00+08:00                  30       4990.000000            4990.0  4990.000000      4990.0   0.000000,                    cmdb_id  total_runs         earliest_run_start
0               frontend-0         100  2022-03-20 12:30:00+08:00
1               frontend-1         100  2022-03-20 12:30:00+08:00
2               frontend-2          78  2022-03-20 12:30:00+08:00
3                adservice         331  2022-03-20 12:30:00+08:00
4               adservice2         316  2022-03-20 12:30:00+08:00
5  productcatalogservice-0          72  2022-03-20 12:30:00+08:00
6  productcatalogservice-1          72  2022-03-20 12:30:00+08:00
7  productcatalogservice-2          78  2022-03-20 12:30:00+08:00
8       shippingservice2-0          78  2022-03-20 12:30:00+08:00)```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse metric_df if present, else load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

# Ensure dtypes
metric_df['timestamp'] = metric_df['timestamp'].astype(int)
metric_df['cmdb_id'] = metric_df['cmdb_id'].astype(str)
metric_df['kpi_name'] = metric_df['kpi_name'].astype(str)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# 1) Compute global P95 and P05 per (cmdb_id, kpi_name) using the full series (no pre-filter)
thresholds_metric = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], observed=True)['value']
    .agg(global_P95=lambda x: x.quantile(0.95), global_P05=lambda x: x.quantile(0.05))
    .reset_index()
)

# 2) Filter metric.csv to the incident window (inclusive) in Asia/Shanghai -> convert to UTC epoch seconds
start_ts = int(pd.Timestamp("2022-03-20T12:30:00+08:00", tz=tz).tz_convert('UTC').timestamp())
end_ts   = int(pd.Timestamp("2022-03-20T12:59:59+08:00", tz=tz).tz_convert('UTC').timestamp())

window_metric = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

# Attach Asia/Shanghai minute string for grouping
window_metric['_dt_shanghai'] = pd.to_datetime(window_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
window_metric['_minute_shanghai'] = window_metric['_dt_shanghai'].dt.floor('min').astype(str)

# 3) Merge thresholds and mark anomalies (using precomputed global thresholds)
window_metric = window_metric.merge(thresholds_metric, on=['cmdb_id', 'kpi_name'], how='left')

# Mark anomaly only when thresholds exist (non-null) and value is not NaN
window_metric['anomaly'] = (~window_metric['global_P95'].isna()) & (~window_metric['value'].isna()) & (
    (window_metric['value'] >= window_metric['global_P95']) | (window_metric['value'] <= window_metric['global_P05'])
)

# 4) Aggregate anomalous rows by minute to produce time series of total anomaly_count per minute
# Create full list of minutes (30 minutes) to ensure zero-count minutes are present
minutes_range = pd.date_range(
    start=pd.Timestamp("2022-03-20T12:30:00", tz=tz),
    end=pd.Timestamp("2022-03-20T12:59:00", tz=tz),
    freq='1min'
)
minutes_df = pd.DataFrame({'_minute_shanghai': minutes_range.astype(str)})

anom_by_min = (
    window_metric[window_metric['anomaly']]
    .groupby('_minute_shanghai', observed=True)
    .size()
    .reset_index(name='total_anomaly_count')
)

minute_agg_df = minutes_df.merge(anom_by_min, on='_minute_shanghai', how='left').fillna({'total_anomaly_count': 0})
minute_agg_df['total_anomaly_count'] = minute_agg_df['total_anomaly_count'].astype(int)

# 5) For the two error-log peak minutes, compute requested details
target_minutes = [
    "2022-03-20 12:45:00+08:00",
    "2022-03-20 12:49:00+08:00"
]

# Prepare containers for outputs
minute_top10_cmdb = {}
minute_top_kpis = {}

for minute in target_minutes:
    # Filter anomalies in that minute
    df_min = window_metric[window_metric['_minute_shanghai'] == minute].copy()
    df_min_anom = df_min[df_min['anomaly']].copy()
    # a) top 10 cmdb_id by anomaly_count in that minute
    top10_cmdb = (
        df_min_anom
        .groupby('cmdb_id', observed=True)
        .size()
        .reset_index(name='anomaly_count')
        .sort_values(by='anomaly_count', ascending=False)
        .head(10)
        .reset_index(drop=True)
    )
    minute_top10_cmdb[minute] = top10_cmdb

    # b) for each of those cmdb_ids, up to top 5 KPIs with aggregates in that minute
    if not top10_cmdb.empty:
        top_cmdb_list = top10_cmdb['cmdb_id'].tolist()
        kpi_agg = (
            df_min_anom[df_min_anom['cmdb_id'].isin(top_cmdb_list)]
            .groupby(['cmdb_id', 'kpi_name'], observed=True)
            .agg(
                anomaly_count_in_minute=('anomaly', 'sum'),
                earliest_anomaly_timestamp_in_minute=('timestamp', 'min'),
                max_value_in_minute=('value', 'max')
            )
            .reset_index()
        )
        # attach global thresholds
        kpi_agg = kpi_agg.merge(thresholds_metric, on=['cmdb_id', 'kpi_name'], how='left')
        # format earliest timestamp to Asia/Shanghai string
        kpi_agg['earliest_anomaly_timestamp_in_minute'] = pd.to_datetime(
            kpi_agg['earliest_anomaly_timestamp_in_minute'], unit='s', utc=True
        ).dt.tz_convert(tz).astype(str)
        # For each cmdb_id, pick top 5 KPIs by anomaly_count_in_minute
        kpi_top_rows = []
        for cmdb in top_cmdb_list:
            sub = kpi_agg[kpi_agg['cmdb_id'] == cmdb].sort_values(by='anomaly_count_in_minute', ascending=False).head(5)
            if not sub.empty:
                kpi_top_rows.append(sub)
        if kpi_top_rows:
            minute_top_kpis[minute] = pd.concat(kpi_top_rows, ignore_index=True)
        else:
            minute_top_kpis[minute] = pd.DataFrame(columns=[
                'cmdb_id','kpi_name','anomaly_count_in_minute','earliest_anomaly_timestamp_in_minute',
                'max_value_in_minute','global_P95','global_P05'
            ])
    else:
        # empty
        minute_top_kpis[minute] = pd.DataFrame(columns=[
            'cmdb_id','kpi_name','anomaly_count_in_minute','earliest_anomaly_timestamp_in_minute',
            'max_value_in_minute','global_P95','global_P05'
        ])

# Final compact outputs:
# - minute_agg_df: 30 rows (minute_timestamp_Asia_Shanghai, total_anomaly_count)
# - minute_top10_cmdb: dict with two DataFrames (top 10 cmdb_id per target minute)
# - minute_top_kpis: dict with two DataFrames (up to 10*5 rows per minute as requested)

minute_agg_df, minute_top10_cmdb, minute_top_kpis

```
Out[6]:
```
```python
# Summarize minute-level anomaly aggregates and details for the two peak error-log minutes.
# Reuse cached variables: minute_agg_df, minute_top10_cmdb, minute_top_kpis.

try:
    minute_agg_df
    minute_top10_cmdb
    minute_top_kpis
except NameError:
    raise RuntimeError("Expected variables minute_agg_df, minute_top10_cmdb, minute_top_kpis to be present in the kernel.")

# Compute basic stats for the 30-minute series
min_count = int(minute_agg_df['total_anomaly_count'].min())
max_count = int(minute_agg_df['total_anomaly_count'].max())
mean_count = float(minute_agg_df['total_anomaly_count'].mean())

# Prepare concise per-minute summary lines (30 rows omitted here; stats provided)
lines = [
    "Minute-level anomaly summary (window: 2022-03-20 12:30 → 12:59:00 +08):",
    f"- Total minutes: {len(minute_agg_df)}",
    f"- Anomalies per minute: min = {min_count}, max = {max_count}, mean ≈ {mean_count:.1f}",
    "",
    "Top components for the two error-log peak minutes and representative top KPIs:",
]

target_minutes = ["2022-03-20 12:45:00+08:00", "2022-03-20 12:49:00+08:00"]

for minute in target_minutes:
    lines.append(f"\nMinute: {minute}")
    top_cmdb = minute_top10_cmdb.get(minute)
    if top_cmdb is None or top_cmdb.empty:
        lines.append("  - No anomalous components in this minute.")
        continue
    # show top 5 components (of the top10) for brevity
    lines.append("  - Top components by anomaly_count (top 5 shown):")
    for idx, row in top_cmdb.head(5).iterrows():
        lines.append(f"    {idx+1}. {row['cmdb_id']} — anomalies: {int(row['anomaly_count'])}")
    # For each of those top 5 components show up to top 3 KPIs with small aggregates
    kpi_df = minute_top_kpis.get(minute)
    if kpi_df is None or kpi_df.empty:
        lines.append("  - No KPI-level anomalies recorded for these components in this minute.")
        continue
    lines.append("  - Representative KPIs for each top component (up to 3 each):")
    for cmdb_row in top_cmdb.head(5).itertuples(index=False):
        cmdb = cmdb_row.cmdb_id
        sub = kpi_df[kpi_df['cmdb_id'] == cmdb].sort_values(by='anomaly_count_in_minute', ascending=False).head(3)
        if sub.empty:
            lines.append(f"    * {cmdb}: (no KPI anomalies listed)")
            continue
        kpis = []
        for r in sub.itertuples(index=False):
            kpis.append(f"{r.kpi_name} (count={int(r.anomaly_count_in_minute)}, max={r.max_value_in_minute}, P95={r.global_P95}, P05={r.global_P05})")
        lines.append(f"    * {cmdb}: " + "; ".join(kpis))

# Final note
lines += [
    "",
    "Overall interpretation:",
    "- Across the 30-minute window total anomaly counts per minute are high (≈3.3k–3.5k), indicating widespread metric values at or beyond global percentiles.",
    "- For both peak error-log minutes (12:45 and 12:49) the top anomaly sources are adservice and adservice2 (≈303 and 293–294 anomalies), followed by several frontend and checkout/productcatalogservice pods.",
    "- At KPI level, many adservice anomalies are per-application traces/metrics (app.http/app.grpc and JVM thread metrics); productcatalogservice and frontend anomalies include container-level KPIs (file descriptors, CPU cfs metrics, sockets, etc.).",
    "",
    "If you want the explicit 30-row minute series or the full top-10 × top-5 KPI tables for each peak minute returned as compact DataFrames, I can provide them."
]

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(             _minute_shanghai  total_anomaly_count
0   2022-03-20 12:30:00+08:00                 3453
1   2022-03-20 12:31:00+08:00                 3367
2   2022-03-20 12:32:00+08:00                 3345
3   2022-03-20 12:33:00+08:00                 3469
4   2022-03-20 12:34:00+08:00                 3380
..                        ...                  ...
25  2022-03-20 12:55:00+08:00                 3293
26  2022-03-20 12:56:00+08:00                 3350
27  2022-03-20 12:57:00+08:00                 3374
28  2022-03-20 12:58:00+08:00                 3545
29  2022-03-20 12:59:00+08:00                 3545

[30 rows x 2 columns], {'2022-03-20 12:45:00+08:00':                    cmdb_id  anomaly_count
0                adservice            303
1               adservice2            293
2        checkoutservice-0            109
3              frontend2-0            101
4               frontend-2             96
5               frontend-0             95
6               frontend-1             94
7       checkoutservice2-0             90
8        checkoutservice-2             78
9  productcatalogservice-1             77, '2022-03-20 12:49:00+08:00':                    cmdb_id  anomaly_count
0                adservice            303
1               adservice2            294
2               frontend-0            108
3              frontend2-0             96
4               frontend-1             90
5        checkoutservice-1             80
6               frontend-2             73
7        checkoutservice-2             72
8  productcatalogservice-0             72
9  productcatalogservice-2             72}, {'2022-03-20 12:45:00+08:00':                     cmdb_id                                           kpi_name  anomaly_count_in_minute earliest_anomaly_timestamp_in_minute  max_value_in_minute  global_P95  global_P05
0                 adservice                  runtime.jvm_threads_state.WAITING                        1            2022-03-20 12:45:00+08:00                 7.00       7.000      7.0000
1                 adservice                                        app.grpc.rr                        1            2022-03-20 12:45:00+08:00               100.00     100.000    100.0000
2                 adservice                                     app.http.count                        1            2022-03-20 12:45:00+08:00                24.00      24.000     24.0000
3                 adservice                                        app.http.rr                        1            2022-03-20 12:45:00+08:00               100.00     100.000    100.0000
4                 adservice                                        app.http.sr                        1            2022-03-20 12:45:00+08:00               100.00     100.000    100.0000
..                      ...                                                ...                      ...                                  ...                  ...         ...         ...
45  productcatalogservice-1         container.node-5.container_cpu_cfs_periods                        1            2022-03-20 12:45:00+08:00               440.50     426.525    332.2500
46  productcatalogservice-1  container.node-5.container_cpu_cfs_throttled_p...                        1            2022-03-20 12:45:00+08:00                 0.00       0.000      0.0000
47  productcatalogservice-1  container.node-5.container_cpu_cfs_throttled_s...                        1            2022-03-20 12:45:00+08:00                 0.00       0.000      0.0000
48  productcatalogservice-1    container.node-5.container_cpu_load_average_10s                        1            2022-03-20 12:45:00+08:00                 0.00       0.000      0.0000
49  productcatalogservice-1      container.node-5.container_cpu_system_seconds                        1            2022-03-20 12:45:00+08:00                 0.46       0.460      0.3345

[50 rows x 7 columns], '2022-03-20 12:49:00+08:00':                     cmdb_id                                           kpi_name  anomaly_count_in_minute earliest_anomaly_timestamp_in_minute  max_value_in_minute  global_P95  global_P05
0                 adservice                  runtime.jvm_threads_state.WAITING                        1            2022-03-20 12:49:00+08:00                  7.0         7.0         7.0
1                 adservice                                        app.grpc.rr                        1            2022-03-20 12:49:00+08:00                100.0       100.0       100.0
2                 adservice                                     app.http.count                        1            2022-03-20 12:49:00+08:00                 24.0        24.0        24.0
3                 adservice                                        app.http.rr                        1            2022-03-20 12:49:00+08:00                100.0       100.0       100.0
4                 adservice                                        app.http.sr                        1            2022-03-20 12:49:00+08:00                100.0       100.0       100.0
..                      ...                                                ...                      ...                                  ...                  ...         ...         ...
45  productcatalogservice-2  container.node-5.container_cpu_cfs_throttled_p...                        1            2022-03-20 12:49:00+08:00                  0.0         0.0         0.0
46  productcatalogservice-2  container.node-5.container_cpu_cfs_throttled_s...                        1            2022-03-20 12:49:00+08:00                  0.0         0.0         0.0
47  productcatalogservice-2    container.node-5.container_cpu_load_average_10s                        1            2022-03-20 12:49:00+08:00                  0.0         0.0         0.0
48  productcatalogservice-2        container.node-5.container_file_descriptors                        1            2022-03-20 12:49:00+08:00                 10.0        10.0        10.0
49  productcatalogservice-2     container.node-5.container_fs_inodes./dev/vda1                        1            2022-03-20 12:49:00+08:00                  0.0         0.0         0.0

[50 rows x 7 columns]})

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# 1) Read metric.csv (no global thresholds computed here)
metric_df = pd.read_csv("metric.csv")

# Ensure dtypes
metric_df['timestamp'] = metric_df['timestamp'].astype(int)
metric_df['cmdb_id'] = metric_df['cmdb_id'].astype(str)
metric_df['kpi_name'] = metric_df['kpi_name'].astype(str)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Target minutes (Asia/Shanghai)
target_minutes = ["2022-03-20 12:45:00+08:00", "2022-03-20 12:49:00+08:00"]

def analyze_minute(minute_str):
    # Parse minute T in Asia/Shanghai and compute epoch seconds for T and T-1
    T = pd.Timestamp(minute_str, tz=tz)
    T_utc_ts = int(T.tz_convert('UTC').timestamp())
    prev_utc_ts = T_utc_ts - 60

    # 2) Filter rows that are at T or T-1
    df_two = metric_df[metric_df['timestamp'].isin([prev_utc_ts, T_utc_ts])].copy()

    # Pivot so each (cmdb_id,kpi_name) has columns for prev and curr if present
    pivot = df_two.pivot_table(
        index=['cmdb_id', 'kpi_name'],
        columns='timestamp',
        values='value',
        aggfunc='first'
    ).reset_index()

    # Rename columns for clarity if present
    col_map = {}
    if prev_utc_ts in pivot.columns:
        col_map[prev_utc_ts] = 'value_prev'
    if T_utc_ts in pivot.columns:
        col_map[T_utc_ts] = 'value_curr'
    pivot = pivot.rename(columns=col_map)

    # Keep only rows that have both prev and curr
    if 'value_prev' not in pivot.columns or 'value_curr' not in pivot.columns:
        # No matching pairs
        empty_A = pd.DataFrame(columns=['cmdb_id','kpi_name','value_prev','value_curr','delta','rel_change'])
        empty_B = empty_A.copy()
        empty_C = pd.DataFrame(columns=['cmdb_id','sum_abs_delta','num_kpis_changed','top_kpi_by_abs_delta'])
        return empty_A, empty_B, empty_C

    df_pairs = pivot.dropna(subset=['value_prev','value_curr']).copy()

    # 2) Compute delta and rel_change
    df_pairs['delta'] = df_pairs['value_curr'] - df_pairs['value_prev']
    # rel_change as specified
    def rel_change_calc(row):
        prev = row['value_prev']
        delta = row['delta']
        if prev == 0 or pd.isna(prev):
            return delta
        else:
            return delta / abs(prev)
    df_pairs['rel_change'] = df_pairs.apply(rel_change_calc, axis=1)

    # A) Top 20 by absolute delta
    df_pairs['abs_delta'] = df_pairs['delta'].abs()
    topA = df_pairs.sort_values(by='abs_delta', ascending=False).head(20)[
        ['cmdb_id','kpi_name','value_prev','value_curr','delta','rel_change']
    ].reset_index(drop=True)

    # B) Top 20 by absolute relative change
    df_pairs['abs_rel_change'] = df_pairs['rel_change'].abs()
    topB = df_pairs.sort_values(by='abs_rel_change', ascending=False).head(20)[
        ['cmdb_id','kpi_name','value_prev','value_curr','delta','rel_change']
    ].reset_index(drop=True)

    # C) Component-level aggregation: sum_abs_delta, num_kpis_changed, top_kpi_by_abs_delta
    comp = df_pairs.groupby('cmdb_id', observed=True).agg(
        sum_abs_delta = ('abs_delta', 'sum'),
        num_kpis_changed = ('kpi_name', 'nunique')
    ).reset_index()

    # Determine top_kpi_by_abs_delta per component
    top_kpi = (
        df_pairs.sort_values(by='abs_delta', ascending=False)
        .groupby('cmdb_id', observed=True)
        .first()
        .reset_index()[['cmdb_id','kpi_name']]
        .rename(columns={'kpi_name':'top_kpi_by_abs_delta'})
    )

    comp = comp.merge(top_kpi, on='cmdb_id', how='left')
    comp = comp.sort_values(by='sum_abs_delta', ascending=False).head(10).reset_index(drop=True)

    # Keep numeric formatting compact
    # Return DataFrames limited as requested
    return topA, topB, comp

# Run analysis for each target minute and collect into dictionary
results = {}
for m in target_minutes:
    A,B,C = analyze_minute(m)
    results[m] = {
        'top20_by_abs_delta': A,
        'top20_by_abs_rel_change': B,
        'top10_components_by_sum_abs_delta': C
    }

# Display the compact structured results dictionary (DataFrames as values)
results

```
Out[7]:
```
```python
# Summarize raw-delta analysis for the two target minutes using cached 'results' variable.
# This is a plain-English summary string (compact).

try:
    results
except NameError:
    raise RuntimeError("Expected 'results' to be present in the kernel.")

def top_names(df, n=3, col='cmdb_id'):
    if df is None or df.empty:
        return []
    return df[col].head(n).astype(str).tolist()

m1 = "2022-03-20 12:45:00+08:00"
m2 = "2022-03-20 12:49:00+08:00"

A1 = results[m1]['top20_by_abs_delta']
B1 = results[m1]['top20_by_abs_rel_change']
C1 = results[m1]['top10_components_by_sum_abs_delta']

A2 = results[m2]['top20_by_abs_delta']
B2 = results[m2]['top20_by_abs_rel_change']
C2 = results[m2]['top10_components_by_sum_abs_delta']

summary_lines = [
    "Raw delta summary for target minutes (values at T vs T-1; no global thresholds used):",
    "",
    f"Minute {m1}:",
    "- Top absolute deltas (representative top 3 KPI combos):"
]

# Representative top-3 absolute delta KPIs for m1
for i, row in A1.head(3).iterrows():
    summary_lines.append(f"  * {row['cmdb_id']} | {row['kpi_name']} — prev={row['value_prev']:.6g}, curr={row['value_curr']:.6g}, delta={row['delta']:.6g}")

summary_lines += [
    "- Top relative-change spikes (representative top 3):"
]
for i, row in B1.head(3).iterrows():
    summary_lines.append(f"  * {row['cmdb_id']} | {row['kpi_name']} — rel_change={row['rel_change']:.6g}, prev={row['value_prev']:.6g}, curr={row['value_curr']:.6g}")

summary_lines += [
    "- Top components by summed absolute delta (top 3):"
]
for i, row in C1.head(3).iterrows():
    summary_lines.append(f"  * {row['cmdb_id']} — sum_abs_delta={row['sum_abs_delta']:.6g}, num_kpis_changed={int(row['num_kpis_changed'])}, top_kpi={row['top_kpi_by_abs_delta']}")

summary_lines += ["", f"Minute {m2}:" , "- Top absolute deltas (representative top 3 KPI combos):"]
for i, row in A2.head(3).iterrows():
    summary_lines.append(f"  * {row['cmdb_id']} | {row['kpi_name']} — prev={row['value_prev']:.6g}, curr={row['value_curr']:.6g}, delta={row['delta']:.6g}")

summary_lines += ["- Top relative-change spikes (representative top 3):"]
for i, row in B2.head(3).iterrows():
    summary_lines.append(f"  * {row['cmdb_id']} | {row['kpi_name']} — rel_change={row['rel_change']:.6g}, prev={row['value_prev']:.6g}, curr={row['value_curr']:.6g}")

summary_lines += ["- Top components by summed absolute delta (top 3):"]
for i, row in C2.head(3).iterrows():
    summary_lines.append(f"  * {row['cmdb_id']} — sum_abs_delta={row['sum_abs_delta']:.6g}, num_kpis_changed={int(row['num_kpis_changed'])}, top_kpi={row['top_kpi_by_abs_delta']}")

summary_lines += [
    "",
    "Key observations and interpretation:",
    "- Absolute-delta lists are dominated by JVM process CPU-time and other monotonic counters (e.g., runtime.java_lang_OperatingSystem_ProcessCpuTime). Large absolute deltas on such counters reflect normal accumulation over time (high rate), not necessarily faults by themselves.",
    "- Memory-related deltas (heap usage, RSS) and container memory fail counts show substantial absolute changes for some services (adservice/adservice2, currencyservice2), suggesting notable memory activity during these minutes.",
    "- Relative-change ranking highlights different signals: e.g., throttling and mesh/traffic counters and some I/O/CPU throttling metrics exhibit very large proportional jumps (checkoutservice / checkoutservice2, node load, mesh metrics). These are good candidates for immediate attention because high relative change can indicate sudden behavior shifts.",
    "- Component-level aggregates show adservice and adservice2 dominate sum_abs_delta in both minutes (they have the largest total changes across many KPIs). Nodes and certain service pods (currencyservice2, checkout/payment-related services) also appear in top component lists.",
    "",
    "Practical next steps:",
    "- Treat large absolute deltas on cumulative counters (CPU time) as rate signals; convert to per-second rates to interpret if they are anomalous relative to normal rates.",
    "- Prioritize investigation on components that appear in both high sum_abs_delta and high relative-change lists (adservice/adservice2, checkoutservice2-related mesh spikes, currencyservice2).",
    "- Correlate these KPI-level deltas with the error-log peaks and trace anomalies previously observed (e.g., frontend errors at 12:45 & 12:49) to find causal links between traffic spikes, throttling, and application/resource metrics.",
    "",
    "If you want the exact DataFrame slices (A:20, B:20, C:10) for either minute as compact tables, I can return them next."
]

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

{'2022-03-20 12:45:00+08:00': {'top20_by_abs_delta': timestamp             cmdb_id                                           kpi_name    value_prev    value_curr         delta  rel_change
0                   adservice   runtime.java_lang_OperatingSystem_ProcessCpuTime  7.450675e+11  7.452642e+11  1.966667e+08    0.000264
1                  adservice2   runtime.java_lang_OperatingSystem_ProcessCpuTime  7.625800e+11  7.627692e+11  1.891667e+08    0.000248
2                  adservice2  runtime.java_lang_Threading_CurrentThreadUserTime  6.826250e+10  6.829917e+10  3.666667e+07    0.000537
3                  adservice2   runtime.java_lang_Threading_CurrentThreadCpuTime  7.125897e+10  7.129531e+10  3.634049e+07    0.000510
4                   adservice  runtime.java_lang_Threading_CurrentThreadUserTime  6.845417e+10  6.842333e+10 -3.083333e+07   -0.000450
5                   adservice   runtime.java_lang_Threading_CurrentThreadCpuTime  7.037623e+10  7.036073e+10 -1.550301e+07   -0.000220
6                  adservice2  runtime.java_lang_Threading_CurrentThreadAlloc...  5.557189e+10  5.558739e+10  1.549318e+07    0.000279
7                   adservice  runtime.java_lang_Threading_CurrentThreadAlloc...  5.568838e+10  5.570231e+10  1.392990e+07    0.000250
8          currencyservice2-0          container.node-6.container_memory_failcnt  6.408081e+07  6.610515e+07  2.024336e+06    0.031590
9                      node-4                              system.fs.inodes.free  6.691750e+09  6.690793e+09 -9.570987e+05   -0.000143
10                     node-4                             system.fs.inodes.total  6.694334e+09  6.693377e+09 -9.570987e+05   -0.000143
11                     node-1                              system.fs.inodes.free  5.028651e+09  5.027920e+09 -7.311360e+05   -0.000145
12                     node-1                             system.fs.inodes.total  5.030588e+09  5.029857e+09 -7.311360e+05   -0.000145
13           paymentservice-2              container.node-5.container_memory_rss  3.229286e+07  3.301171e+07  7.188480e+05    0.022260
14                     node-2                              system.fs.inodes.free  5.028617e+09  5.027903e+09 -7.142400e+05   -0.000142
15                     node-2                             system.fs.inodes.total  5.030584e+09  5.029871e+09 -7.137280e+05   -0.000142
16                 adservice2      runtime.java_lang_Memory_HeapMemoryUsage_used  2.078499e+07  2.143064e+07  6.456453e+05    0.031063
17                 adservice2  runtime.java_lang_MemoryPool_Usage_used.Eden_S...  3.803467e+06  4.439693e+06  6.362253e+05    0.167275
18         currencyservice2-0      container.node-6.container_memory_mapped_file  7.434240e+05  1.624747e+05 -5.809493e+05   -0.781451
19         checkoutservice2-0              container.node-5.container_memory_rss  1.574093e+07  1.516749e+07 -5.734400e+05   -0.036430, 'top20_by_abs_rel_change': timestamp             cmdb_id                                           kpi_name  value_prev   value_curr        delta  rel_change
0           checkoutservice-2  container.node-5.container_cpu_cfs_throttled_s...    0.009641     0.087490     0.077849    8.074910
1                      node-3                                      system.load.1    0.220000     1.400000     1.180000    5.363636
2          checkoutservice2-0  mesh.source.checkoutservice2.jaeger-collector....    1.500000     8.000000     6.500000    4.333333
3           checkoutservice-2  container.node-5.container_cpu_cfs_throttled_p...    0.500000     2.500000     2.000000    4.000000
4           paymentservice2-0  mesh.destination.checkoutservice2.paymentservi...  670.000000  2700.000000  2030.000000    3.029851
5           currencyservice-0  container.node-5.container_network_receive_MB....    0.176315     0.708027     0.531712    3.015686
6                  frontend-1  mesh.source.frontend.jaeger-collector.istio_re...    0.003150     0.012600     0.009450    3.000000
7          checkoutservice2-0  mesh.source.checkoutservice2.currencyservice2....    3.000000    12.000000     9.000000    3.000000
8                      node-6                                      system.io.w_s    0.000000     3.000000     3.000000    3.000000
9          checkoutservice2-0  mesh.source.checkoutservice2.emailservice2.ist...    1.000000     4.000000     3.000000    3.000000
10         checkoutservice2-0  mesh.source.checkoutservice2.emailservice2.ist...    1.000000     4.000000     3.000000    3.000000
11         checkoutservice2-0  mesh.source.checkoutservice2.emailservice2.ist...    1.000000     4.000000     3.000000    3.000000
12         checkoutservice2-0  mesh.source.checkoutservice2.paymentservice2.i...    1.000000     4.000000     3.000000    3.000000
13         checkoutservice2-0  mesh.source.checkoutservice2.currencyservice2....    3.000000    12.000000     9.000000    3.000000
14         checkoutservice2-0  mesh.source.checkoutservice2.paymentservice2.i...    1.000000     4.000000     3.000000    3.000000
15         checkoutservice2-0  mesh.source.checkoutservice2.shippingservice2....    2.000000     8.000000     6.000000    3.000000
16         checkoutservice2-0  mesh.source.checkoutservice2.shippingservice2....    2.000000     8.000000     6.000000    3.000000
17         checkoutservice2-0  mesh.source.checkoutservice2.shippingservice2....    2.000000     8.000000     6.000000    3.000000
18         checkoutservice2-0  mesh.source.checkoutservice2.paymentservice2.i...    1.000000     4.000000     3.000000    3.000000
19         checkoutservice2-0  mesh.source.checkoutservice2.productcatalogser...    2.000000     8.000000     6.000000    3.000000, 'top10_components_by_sum_abs_delta':               cmdb_id  sum_abs_delta  num_kpis_changed                              top_kpi_by_abs_delta
0          adservice2   2.792749e+08               333  runtime.java_lang_OperatingSystem_ProcessCpuTime
1           adservice   2.572844e+08               341  runtime.java_lang_OperatingSystem_ProcessCpuTime
2  currencyservice2-0   2.788475e+06                76         container.node-6.container_memory_failcnt
3              node-4   2.480110e+06                57                             system.fs.inodes.free
4              node-1   2.426408e+06                56                             system.fs.inodes.free
5              node-2   1.855632e+06                57                             system.fs.inodes.free
6          frontend-2   1.519466e+06               146             container.node-5.container_memory_rss
7              node-3   1.301899e+06                56                             system.fs.inodes.free
8              node-6   7.682771e+05                56                            system.fs.inodes.total
9    paymentservice-2   7.200059e+05                70             container.node-5.container_memory_rss}, '2022-03-20 12:49:00+08:00': {'top20_by_abs_delta': timestamp             cmdb_id                                           kpi_name    value_prev    value_curr         delta  rel_change
0                   adservice   runtime.java_lang_OperatingSystem_ProcessCpuTime  7.458600e+11  7.460558e+11  1.958333e+08    0.000263
1                  adservice2   runtime.java_lang_OperatingSystem_ProcessCpuTime  7.633525e+11  7.635383e+11  1.858333e+08    0.000243
2                   adservice  runtime.java_lang_Threading_CurrentThreadUserTime  6.848917e+10  6.854250e+10  5.333333e+07    0.000779
3                   adservice   runtime.java_lang_Threading_CurrentThreadCpuTime  7.042141e+10  7.046773e+10  4.632145e+07    0.000658
4                  adservice2  runtime.java_lang_Threading_CurrentThreadUserTime  6.836833e+10  6.835167e+10 -1.666667e+07   -0.000244
5                   adservice  runtime.java_lang_Threading_CurrentThreadAlloc...  5.574592e+10  5.576153e+10  1.560467e+07    0.000280
6                  adservice2  runtime.java_lang_Threading_CurrentThreadAlloc...  5.563159e+10  5.564501e+10  1.341716e+07    0.000241
7                  adservice2   runtime.java_lang_Threading_CurrentThreadCpuTime  7.136611e+10  7.135353e+10 -1.258025e+07   -0.000176
8          currencyservice2-0          container.node-6.container_memory_failcnt  7.120377e+07  7.267110e+07  1.467325e+06    0.020607
9                      node-4                              system.fs.inodes.free  6.697814e+09  6.696663e+09 -1.150976e+06   -0.000172
10                     node-4                             system.fs.inodes.total  6.700397e+09  6.699247e+09 -1.150293e+06   -0.000172
11                     node-1                             system.fs.inodes.total  5.035137e+09  5.034263e+09 -8.739840e+05   -0.000174
12                     node-2                             system.fs.inodes.total  5.035137e+09  5.034263e+09 -8.739840e+05   -0.000174
13                     node-1                              system.fs.inodes.free  5.033200e+09  5.032326e+09 -8.734720e+05   -0.000174
14                     node-2                              system.fs.inodes.free  5.033170e+09  5.032296e+09 -8.734720e+05   -0.000174
15          checkoutservice-1              container.node-5.container_memory_rss  1.563034e+07  1.487258e+07 -7.577600e+05   -0.048480
16         currencyservice2-0              container.node-6.container_memory_rss  5.410406e+07  5.338180e+07 -7.222613e+05   -0.013349
17         currencyservice2-0            container.node-6.container_memory_cache  6.660506e+07  6.732459e+07  7.195307e+05    0.010803
18          currencyservice-0              container.node-5.container_memory_rss  3.169280e+07  3.102720e+07 -6.656000e+05   -0.021002
19         currencyservice2-0      container.node-6.container_memory_mapped_file  1.142784e+06  5.065387e+05 -6.362453e+05   -0.556750, 'top20_by_abs_rel_change': timestamp                  cmdb_id                                           kpi_name   value_prev    value_curr         delta  rel_change
0               currencyservice2-0  container.node-6.container_cpu_cfs_throttled_s...     0.016766      0.254838      0.238071   14.199507
1                 paymentservice-0  container.node-5.container_network_receive_MB....     0.023876      0.286684      0.262808   11.007110
2               checkoutservice2-0  mesh.source.checkoutservice2.currencyservice2....     0.032375      0.229600      0.197225    6.091892
3                checkoutservice-1  container.node-5.container_network_receive_MB....     0.049139      0.324214      0.275075    5.597902
4                checkoutservice-1  container.node-5.container_cpu_cfs_throttled_s...     0.084368      0.440485      0.356117    4.220992
5               checkoutservice2-0  mesh.destination.frontend2.checkoutservice2.is...     0.073750      0.362000      0.288250    3.908475
6          productcatalogservice-2  mesh.source.productcatalogservice.jaeger-colle...     0.001050      0.004725      0.003675    3.500000
7               checkoutservice2-0  mesh.source.checkoutservice2.paymentservice2.i...     0.001550      0.005675      0.004125    2.661290
8               checkoutservice2-0  mesh.source.checkoutservice2.productcatalogser...   585.000000   2095.000000   1510.000000    2.581197
9               checkoutservice2-0  mesh.source.checkoutservice2.currencyservice2....  4850.000000  17075.000000  12225.000000    2.520619
10              checkoutservice2-0  mesh.source.checkoutservice2.cartservice2.isti...     0.004100      0.014350      0.010250    2.500000
11              checkoutservice2-0  mesh.source.checkoutservice2.cartservice2.isti...     2.000000      7.000000      5.000000    2.500000
12              checkoutservice2-0  mesh.source.checkoutservice2.cartservice2.isti...  3300.000000  11550.000000   8250.000000    2.500000
13              checkoutservice2-0  mesh.destination.frontend2.checkoutservice2.is...     1.000000      3.500000      2.500000    2.500000
14              checkoutservice2-0  mesh.destination.frontend2.checkoutservice2.is...  1450.000000   5075.000000   3625.000000    2.500000
15              checkoutservice2-0  mesh.source.checkoutservice2.cartservice2.isti...     2.000000      7.000000      5.000000    2.500000
16                     frontend2-0  mesh.source.frontend2.checkoutservice2.istio_r...     1.000000      3.500000      2.500000    2.500000
17                     frontend2-0  mesh.source.frontend2.checkoutservice2.istio_r...     1.000000      3.500000      2.500000    2.500000
18                     frontend2-0  mesh.source.frontend2.checkoutservice2.istio_r...     1.000000      3.500000      2.500000    2.500000
19              checkoutservice2-0  mesh.destination.frontend2.checkoutservice2.is...     1.000000      3.500000      2.500000    2.500000, 'top10_components_by_sum_abs_delta':               cmdb_id  sum_abs_delta  num_kpis_changed                              top_kpi_by_abs_delta
0           adservice   3.120418e+08               341  runtime.java_lang_OperatingSystem_ProcessCpuTime
1          adservice2   2.294675e+08               333  runtime.java_lang_OperatingSystem_ProcessCpuTime
2  currencyservice2-0   3.665110e+06                76         container.node-6.container_memory_failcnt
3              node-4   3.011252e+06                57                             system.fs.inodes.free
4              node-1   2.684207e+06                56                            system.fs.inodes.total
5              node-2   2.203883e+06                57                            system.fs.inodes.total
6              node-3   8.774162e+05                56                             system.fs.inodes.free
7   checkoutservice-1   8.370674e+05               118             container.node-5.container_memory_rss
8   currencyservice-0   7.960260e+05                76             container.node-5.container_memory_rss
9          frontend-1   6.566845e+05               156             container.node-5.container_memory_rss}}```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz

# reuse timezone if present
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# reuse trace_df if present, else load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

# Ensure dtypes
trace_df['timestamp'] = trace_df['timestamp'].astype(int)
trace_df['cmdb_id'] = trace_df['cmdb_id'].astype(str)
trace_df['trace_name'] = trace_df['trace_name'].astype(str)
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

target_minutes = ["2022-03-20 12:45:00+08:00", "2022-03-20 12:49:00+08:00"]

def analyze_trace_minute(minute_str):
    T = pd.Timestamp(minute_str, tz=tz)
    T_utc = int(T.tz_convert('UTC').timestamp())
    prev_utc = T_utc - 60

    # select rows at T and T-1
    df_sel = trace_df[trace_df['timestamp'].isin([prev_utc, T_utc])].copy()

    # pivot to have value_prev and value_curr
    pivot = df_sel.pivot_table(
        index=['cmdb_id', 'trace_name'],
        columns='timestamp',
        values='value',
        aggfunc='first'
    ).reset_index()

    col_map = {}
    if prev_utc in pivot.columns:
        col_map[prev_utc] = 'value_prev'
    if T_utc in pivot.columns:
        col_map[T_utc] = 'value_curr'
    pivot = pivot.rename(columns=col_map)

    # keep only rows with both values
    if ('value_prev' not in pivot.columns) or ('value_curr' not in pivot.columns):
        empty_cols = ['cmdb_id','trace_name','value_prev','value_curr','delta','rel_change']
        empty_df = pd.DataFrame(columns=empty_cols)
        return {
            'top20_by_abs_delta': empty_df,
            'top20_from_frontend_by_abs_delta': empty_df,
            'top20_by_abs_rel_change': empty_df,
            'top10_components': pd.DataFrame(columns=['cmdb_id','sum_abs_delta','num_traces_changed','top_trace_by_abs_delta'])
        }

    df_pairs = pivot.dropna(subset=['value_prev','value_curr']).copy()
    if df_pairs.empty:
        empty_cols = ['cmdb_id','trace_name','value_prev','value_curr','delta','rel_change']
        empty_df = pd.DataFrame(columns=empty_cols)
        return {
            'top20_by_abs_delta': empty_df,
            'top20_from_frontend_by_abs_delta': empty_df,
            'top20_by_abs_rel_change': empty_df,
            'top10_components': pd.DataFrame(columns=['cmdb_id','sum_abs_delta','num_traces_changed','top_trace_by_abs_delta'])
        }

    # compute delta and rel_change
    df_pairs['delta'] = df_pairs['value_curr'] - df_pairs['value_prev']

    def rel_change_calc(row):
        prev = row['value_prev']
        delta = row['delta']
        if pd.isna(prev) or prev == 0:
            return delta
        return delta / abs(prev)

    df_pairs['rel_change'] = df_pairs.apply(rel_change_calc, axis=1)
    df_pairs['abs_delta'] = df_pairs['delta'].abs()
    df_pairs['abs_rel_change'] = df_pairs['rel_change'].abs()

    # 1) Top20 by absolute delta
    top20_abs_delta = df_pairs.sort_values(by='abs_delta', ascending=False).head(20)[
        ['cmdb_id','trace_name','value_prev','value_curr','delta','rel_change']
    ].reset_index(drop=True)

    # 2) Top20 among trace_name containing 'from_frontend' by abs delta
    mask_front = df_pairs['trace_name'].str.contains('from_frontend', na=False)
    df_front = df_pairs[mask_front]
    top20_front = df_front.sort_values(by='abs_delta', ascending=False).head(20)[
        ['cmdb_id','trace_name','value_prev','value_curr','delta','rel_change']
    ].reset_index(drop=True)

    # 3) Top20 by absolute relative change
    top20_abs_rel = df_pairs.sort_values(by='abs_rel_change', ascending=False).head(20)[
        ['cmdb_id','trace_name','value_prev','value_curr','delta','rel_change']
    ].reset_index(drop=True)

    # 4) Component-level aggregation
    comp = df_pairs.groupby('cmdb_id', observed=True).agg(
        sum_abs_delta = ('abs_delta', 'sum'),
        num_traces_changed = ('trace_name', 'nunique')
    ).reset_index()

    # determine top_trace_by_abs_delta per component
    top_trace = df_pairs.sort_values(by='abs_delta', ascending=False).groupby('cmdb_id', observed=True).first().reset_index()
    if 'trace_name' in top_trace.columns:
        top_trace = top_trace[['cmdb_id','trace_name']].rename(columns={'trace_name':'top_trace_by_abs_delta'})
    else:
        top_trace = top_trace[['cmdb_id']].assign(top_trace_by_abs_delta=None)

    comp = comp.merge(top_trace, on='cmdb_id', how='left')
    comp = comp.sort_values(by='sum_abs_delta', ascending=False).head(10).reset_index(drop=True)

    # keep columns consistent
    comp = comp[['cmdb_id','sum_abs_delta','num_traces_changed','top_trace_by_abs_delta']]

    return {
        'top20_by_abs_delta': top20_abs_delta,
        'top20_from_frontend_by_abs_delta': top20_front,
        'top20_by_abs_rel_change': top20_abs_rel,
        'top10_components': comp
    }

# Run for both minutes and store results
results_trace_deltas = {}
for m in target_minutes:
    results_trace_deltas[m] = analyze_trace_minute(m)

# Return the compact structured dictionary (DataFrames are limited to requested maxima)
results_trace_deltas

```
Out[8]:
```
```python
# Summarize trace delta analysis for the two target minutes using cached results_trace_deltas.
try:
    results_trace_deltas
except NameError:
    raise RuntimeError("Expected 'results_trace_deltas' to be present in the kernel.")

m1 = "2022-03-20 12:45:00+08:00"
m2 = "2022-03-20 12:49:00+08:00"

A1 = results_trace_deltas[m1]['top20_by_abs_delta']
F1 = results_trace_deltas[m1]['top20_from_frontend_by_abs_delta']
R1 = results_trace_deltas[m1]['top20_by_abs_rel_change']
C1 = results_trace_deltas[m1]['top10_components']

A2 = results_trace_deltas[m2]['top20_by_abs_delta']
F2 = results_trace_deltas[m2]['top20_from_frontend_by_abs_delta']
R2 = results_trace_deltas[m2]['top20_by_abs_rel_change']
C2 = results_trace_deltas[m2]['top10_components']

summary_lines = [
    "Trace delta summary for target minutes (values at T vs T-1; delta = T - T-1, rel_change = delta/abs(prev) or delta if prev==0):",
    ""
]

# Minute 12:45 summary
summary_lines += [
    f"Minute {m1}:",
    "- Top absolute deltas (representative top 3):"
]
for i, row in A1.head(3).iterrows():
    summary_lines.append(
        f"  * {row['cmdb_id']} | {row['trace_name']} — prev={row['value_prev']}, curr={row['value_curr']}, delta={row['delta']}, rel_change={row['rel_change']}"
    )

summary_lines += [
    "- Top 'from_frontend' deltas (representative top 3):"
]
for i, row in F1.head(3).iterrows():
    summary_lines.append(
        f"  * {row['cmdb_id']} | {row['trace_name']} — delta={row['delta']}, rel_change={row['rel_change']}"
    )

summary_lines += [
    "- Top relative-change spikes (representative top 3):"
]
for i, row in R1.head(3).iterrows():
    summary_lines.append(
        f"  * {row['cmdb_id']} | {row['trace_name']} — rel_change={row['rel_change']}, prev={row['value_prev']}, curr={row['value_curr']}"
    )

summary_lines += [
    "- Component-level top contributors by sum_abs_delta (top 5):"
]
for i, row in C1.head(5).iterrows():
    summary_lines.append(
        f"  * {row['cmdb_id']} — sum_abs_delta={row['sum_abs_delta']}, num_traces_changed={int(row['num_traces_changed'])}, top_trace={row['top_trace_by_abs_delta']}"
    )

# Minute 12:49 summary
summary_lines += ["", f"Minute {m2}:"]
summary_lines += [
    "- Top absolute deltas (representative top 3):"
]
for i, row in A2.head(3).iterrows():
    summary_lines.append(
        f"  * {row['cmdb_id']} | {row['trace_name']} — prev={row['value_prev']}, curr={row['value_curr']}, delta={row['delta']}, rel_change={row['rel_change']}"
    )

summary_lines += [
    "- Top 'from_frontend' deltas (representative top 3):"
]
for i, row in F2.head(3).iterrows():
    summary_lines.append(
        f"  * {row['cmdb_id']} | {row['trace_name']} — delta={row['delta']}, rel_change={row['rel_change']}"
    )

summary_lines += [
    "- Top relative-change spikes (representative top 3):"
]
for i, row in R2.head(3).iterrows():
    summary_lines.append(
        f"  * {row['cmdb_id']} | {row['trace_name']} — rel_change={row['rel_change']}, prev={row['value_prev']}, curr={row['value_curr']}"
    )

summary_lines += [
    "- Component-level top contributors by sum_abs_delta (top 5):"
]
for i, row in C2.head(5).iterrows():
    summary_lines.append(
        f"  * {row['cmdb_id']} — sum_abs_delta={row['sum_abs_delta']}, num_traces_changed={int(row['num_traces_changed'])}, top_trace={row['top_trace_by_abs_delta']}"
    )

summary_lines += [
    "",
    "Key takeaways:",
    "- The absolute-delta lists are dominated by frontend self-row_count changes (large increases at 12:45 for frontends, decreases or shifts at 12:49). These represent traffic volume changes.",
    "- 'from_frontend' row_count deltas show downstream services (currencyservice, productcatalogservice variants) receiving notably different loads between the two minutes—useful for tracing request flow.",
    "- Relative-change top items highlight many small-count traces that multiplied (e.g., checkoutservice/checkoutservice2 and mesh/source metrics) — these are sudden proportional spikes and good candidates for further inspection.",
    "- Component aggregates show frontends as major contributors to total trace delta (frontends top the per-minute sum_abs_delta lists), with currency/product-related services also significant.",
    "",
    "If you want the exact limited tables (A:20, B:20 filtered 'from_frontend', C:20 by rel_change, and component top10) returned as DataFrames for each minute, I can provide them next."
]

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

{'2022-03-20 12:45:00+08:00': {'top20_by_abs_delta': timestamp                   cmdb_id                                   trace_name  value_prev  value_curr  delta  rel_change
0                        frontend-2                         trace.self.row_count      1176.0      1401.0  225.0    0.191327
1                        frontend-0                         trace.self.row_count      1100.0      1191.0   91.0    0.082727
2                        frontend-1                         trace.self.row_count       888.0       975.0   87.0    0.097973
3                       frontend2-0        trace.to_currencyservice2-0.row_count       164.0       236.0   72.0    0.439024
4                currencyservice2-0             trace.from_frontend2-0.row_count       164.0       236.0   72.0    0.439024
5                       frontend2-0  trace.to_productcatalogservice2-0.row_count       341.0       271.0  -70.0   -0.205279
6          productcatalogservice2-0             trace.from_frontend2-0.row_count       341.0       271.0  -70.0   -0.205279
7                        frontend-2         trace.to_currencyservice-2.row_count       108.0       148.0   40.0    0.370370
8                 currencyservice-2              trace.from_frontend-2.row_count       108.0       148.0   40.0    0.370370
9                        frontend-2         trace.to_currencyservice-1.row_count       109.0       147.0   38.0    0.348624
10                       frontend-2         trace.to_currencyservice-0.row_count       109.0       147.0   38.0    0.348624
11                currencyservice-1              trace.from_frontend-2.row_count       109.0       147.0   38.0    0.348624
12                currencyservice-0              trace.from_frontend-2.row_count       109.0       147.0   38.0    0.348624
13          productcatalogservice-2              trace.from_frontend-0.row_count       162.0       194.0   32.0    0.197531
14          productcatalogservice-0              trace.from_frontend-0.row_count       162.0       194.0   32.0    0.197531
15          productcatalogservice-1              trace.from_frontend-0.row_count       162.0       194.0   32.0    0.197531
16                       frontend-0   trace.to_productcatalogservice-2.row_count       162.0       194.0   32.0    0.197531
17                       frontend-0   trace.to_productcatalogservice-1.row_count       162.0       194.0   32.0    0.197531
18                       frontend-0   trace.to_productcatalogservice-0.row_count       162.0       194.0   32.0    0.197531
19                      frontend2-0                         trace.self.row_count       682.0       654.0  -28.0   -0.041056, 'top20_from_frontend_by_abs_delta': timestamp                   cmdb_id                        trace_name  value_prev  value_curr  delta  rel_change
0                currencyservice2-0  trace.from_frontend2-0.row_count       164.0       236.0   72.0    0.439024
1          productcatalogservice2-0  trace.from_frontend2-0.row_count       341.0       271.0  -70.0   -0.205279
2                 currencyservice-2   trace.from_frontend-2.row_count       108.0       148.0   40.0    0.370370
3                 currencyservice-1   trace.from_frontend-2.row_count       109.0       147.0   38.0    0.348624
4                 currencyservice-0   trace.from_frontend-2.row_count       109.0       147.0   38.0    0.348624
5           productcatalogservice-2   trace.from_frontend-0.row_count       162.0       194.0   32.0    0.197531
6           productcatalogservice-0   trace.from_frontend-0.row_count       162.0       194.0   32.0    0.197531
7           productcatalogservice-1   trace.from_frontend-0.row_count       162.0       194.0   32.0    0.197531
8                 currencyservice-0   trace.from_frontend-1.row_count        79.0       104.0   25.0    0.316456
9                 currencyservice-1   trace.from_frontend-1.row_count        80.0       104.0   24.0    0.300000
10          productcatalogservice-1   trace.from_frontend-2.row_count       186.0       209.0   23.0    0.123656
11                currencyservice-2   trace.from_frontend-1.row_count        80.0       103.0   23.0    0.287500
12          productcatalogservice-2   trace.from_frontend-2.row_count       187.0       208.0   21.0    0.112299
13          productcatalogservice-0   trace.from_frontend-2.row_count       187.0       208.0   21.0    0.112299
14                currencyservice-0   trace.from_frontend-0.row_count       117.0       102.0  -15.0   -0.128205
15                currencyservice-1   trace.from_frontend-0.row_count       117.0       102.0  -15.0   -0.128205
16                currencyservice-2   trace.from_frontend-0.row_count       116.0       102.0  -14.0   -0.120690
17                     adservice2-0  trace.from_frontend2-0.row_count        48.0        34.0  -14.0   -0.291667
18         recommendationservice2-0  trace.from_frontend2-0.row_count        54.0        41.0  -13.0   -0.240741
19                   cartservice2-0  trace.from_frontend2-0.row_count        65.0        52.0  -13.0   -0.200000, 'top20_by_abs_rel_change': timestamp            cmdb_id                                 trace_name  value_prev  value_curr     delta  rel_change
0          checkoutservice-2       trace.to_shippingservice-1.row_count    1.000000    4.000000  3.000000    3.000000
1          shippingservice-1     trace.from_checkoutservice-2.row_count    1.000000    4.000000  3.000000    3.000000
2          checkoutservice-2    trace.to_shippingservice-1.duration_p95    0.000057    0.000149  0.000092    1.614035
3          shippingservice-1  trace.from_checkoutservice-2.duration_p95    0.000057    0.000149  0.000092    1.614035
4          checkoutservice-2       trace.to_currencyservice-1.row_count    2.000000    5.000000  3.000000    1.500000
5          currencyservice-1     trace.from_checkoutservice-2.row_count    2.000000    5.000000  3.000000    1.500000
6                 frontend-1       trace.to_checkoutservice-0.row_count    1.000000    2.000000  1.000000    1.000000
7          checkoutservice-2        trace.to_paymentservice-0.row_count    1.000000    2.000000  1.000000    1.000000
8              cartservice-2     trace.from_checkoutservice-2.row_count    2.000000    4.000000  2.000000    1.000000
9          checkoutservice-0            trace.from_frontend-1.row_count    1.000000    2.000000  1.000000    1.000000
10             cartservice-2         trace.from_frontend-2.duration_p95    0.000300    0.000000 -0.000300   -1.000000
11             cartservice-2        trace.from_frontend-0.duration_mean    0.000031    0.000000 -0.000031   -1.000000
12                frontend-0       trace.to_cartservice-0.duration_mean    0.000032    0.000000 -0.000032   -1.000000
13                frontend-0       trace.to_cartservice-2.duration_mean    0.000031    0.000000 -0.000031   -1.000000
14                frontend-2       trace.to_checkoutservice-2.row_count    1.000000    2.000000  1.000000    1.000000
15          paymentservice-1     trace.from_checkoutservice-2.row_count    1.000000    2.000000  1.000000    1.000000
16         checkoutservice-0        trace.to_paymentservice-0.row_count    1.000000    2.000000  1.000000    1.000000
17         checkoutservice-2            trace.from_frontend-2.row_count    1.000000    2.000000  1.000000    1.000000
18         checkoutservice-2        trace.to_paymentservice-1.row_count    1.000000    2.000000  1.000000    1.000000
19                frontend-1       trace.to_checkoutservice-1.row_count    1.000000    2.000000  1.000000    1.000000, 'top10_components':                    cmdb_id  sum_abs_delta  num_traces_changed                 top_trace_by_abs_delta
0               frontend-2     483.024842                  92                   trace.self.row_count
1               frontend-0     284.041462                  92                   trace.self.row_count
2              frontend2-0     219.040909                  32  trace.to_currencyservice2-0.row_count
3               frontend-1     183.092961                  88                   trace.self.row_count
4        currencyservice-1      81.000114                  24        trace.from_frontend-2.row_count
5        currencyservice-0      81.000066                  24        trace.from_frontend-2.row_count
6        currencyservice-2      78.000051                  24        trace.from_frontend-2.row_count
7  productcatalogservice-0      72.004475                  40        trace.from_frontend-0.row_count
8       currencyservice2-0      72.002554                   4       trace.from_frontend2-0.row_count
9  productcatalogservice-1      71.006886                  40        trace.from_frontend-0.row_count}, '2022-03-20 12:49:00+08:00': {'top20_by_abs_delta': timestamp                   cmdb_id                                   trace_name  value_prev  value_curr  delta  rel_change
0                        frontend-2                         trace.self.row_count      1323.0      1246.0  -77.0   -0.058201
1                       frontend2-0        trace.to_currencyservice2-0.row_count       157.0       229.0   72.0    0.458599
2                currencyservice2-0             trace.from_frontend2-0.row_count       157.0       229.0   72.0    0.458599
3                        frontend-1                         trace.self.row_count       963.0       904.0  -59.0   -0.061267
4                 currencyservice-1              trace.from_frontend-0.row_count       145.0       101.0  -44.0   -0.303448
5                        frontend-0         trace.to_currencyservice-1.row_count       145.0       101.0  -44.0   -0.303448
6                 currencyservice-2              trace.from_frontend-0.row_count       145.0       101.0  -44.0   -0.303448
7                        frontend-0         trace.to_currencyservice-0.row_count       145.0       101.0  -44.0   -0.303448
8                        frontend-0         trace.to_currencyservice-2.row_count       145.0       101.0  -44.0   -0.303448
9                 currencyservice-0              trace.from_frontend-0.row_count       145.0       101.0  -44.0   -0.303448
10                      frontend2-0  trace.to_productcatalogservice2-0.row_count       331.0       293.0  -38.0   -0.114804
11         productcatalogservice2-0             trace.from_frontend2-0.row_count       331.0       293.0  -38.0   -0.114804
12          productcatalogservice-0              trace.from_frontend-0.row_count       174.0       206.0   32.0    0.183908
13                       frontend-0   trace.to_productcatalogservice-0.row_count       174.0       206.0   32.0    0.183908
14                checkoutservice-1                         trace.self.row_count        56.0        24.0  -32.0   -0.571429
15                       frontend-0   trace.to_productcatalogservice-2.row_count       174.0       205.0   31.0    0.178161
16          productcatalogservice-2              trace.from_frontend-0.row_count       174.0       205.0   31.0    0.178161
17          productcatalogservice-1              trace.from_frontend-0.row_count       175.0       205.0   30.0    0.171429
18                       frontend-0   trace.to_productcatalogservice-1.row_count       175.0       205.0   30.0    0.171429
19                      frontend2-0                         trace.self.row_count       656.0       681.0   25.0    0.038110, 'top20_from_frontend_by_abs_delta': timestamp                   cmdb_id                        trace_name  value_prev  value_curr  delta  rel_change
0                currencyservice2-0  trace.from_frontend2-0.row_count       157.0       229.0   72.0    0.458599
1                 currencyservice-1   trace.from_frontend-0.row_count       145.0       101.0  -44.0   -0.303448
2                 currencyservice-0   trace.from_frontend-0.row_count       145.0       101.0  -44.0   -0.303448
3                 currencyservice-2   trace.from_frontend-0.row_count       145.0       101.0  -44.0   -0.303448
4          productcatalogservice2-0  trace.from_frontend2-0.row_count       331.0       293.0  -38.0   -0.114804
5           productcatalogservice-0   trace.from_frontend-0.row_count       174.0       206.0   32.0    0.183908
6           productcatalogservice-2   trace.from_frontend-0.row_count       174.0       205.0   31.0    0.178161
7           productcatalogservice-1   trace.from_frontend-0.row_count       175.0       205.0   30.0    0.171429
8           productcatalogservice-1   trace.from_frontend-1.row_count       155.0       133.0  -22.0   -0.141935
9           productcatalogservice-0   trace.from_frontend-1.row_count       155.0       133.0  -22.0   -0.141935
10          productcatalogservice-2   trace.from_frontend-1.row_count       155.0       134.0  -21.0   -0.135484
11          productcatalogservice-0   trace.from_frontend-2.row_count       208.0       189.0  -19.0   -0.091346
12          productcatalogservice-2   trace.from_frontend-2.row_count       208.0       189.0  -19.0   -0.091346
13          productcatalogservice-1   trace.from_frontend-2.row_count       208.0       189.0  -19.0   -0.091346
14                currencyservice-0   trace.from_frontend-1.row_count        85.0        97.0   12.0    0.141176
15                currencyservice-1   trace.from_frontend-1.row_count        86.0        96.0   10.0    0.116279
16                currencyservice-2   trace.from_frontend-1.row_count        86.0        96.0   10.0    0.116279
17          recommendationservice-2   trace.from_frontend-0.row_count        26.0        33.0    7.0    0.269231
18         recommendationservice2-0  trace.from_frontend2-0.row_count        52.0        45.0   -7.0   -0.134615
19                shippingservice-1   trace.from_frontend-2.row_count        12.0         6.0   -6.0   -0.500000, 'top20_by_abs_rel_change': timestamp                   cmdb_id                                   trace_name  value_prev  value_curr      delta  rel_change
0                checkoutservice2-0        trace.to_shippingservice2-0.row_count    2.000000    8.000000   6.000000    3.000000
1                currencyservice2-0      trace.from_checkoutservice2-0.row_count    3.000000   12.000000   9.000000    3.000000
2                checkoutservice2-0                    trace.from_root.row_count    3.000000   12.000000   9.000000    3.000000
3                checkoutservice2-0                         trace.self.row_count    8.000000   32.000000  24.000000    3.000000
4                checkoutservice2-0            trace.to_cartservice2-0.row_count    2.000000    8.000000   6.000000    3.000000
5                checkoutservice2-0           trace.to_emailservice2-0.row_count    1.000000    4.000000   3.000000    3.000000
6                checkoutservice2-0        trace.to_currencyservice2-0.row_count    3.000000   12.000000   9.000000    3.000000
7                checkoutservice2-0             trace.from_frontend2-0.row_count    1.000000    4.000000   3.000000    3.000000
8                   emailservice2-0      trace.from_checkoutservice2-0.row_count    1.000000    4.000000   3.000000    3.000000
9          productcatalogservice2-0      trace.from_checkoutservice2-0.row_count    2.000000    8.000000   6.000000    3.000000
10                   cartservice2-0      trace.from_checkoutservice2-0.row_count    2.000000    8.000000   6.000000    3.000000
11                      frontend2-0        trace.to_checkoutservice2-0.row_count    1.000000    4.000000   3.000000    3.000000
12               shippingservice2-0      trace.from_checkoutservice2-0.row_count    2.000000    8.000000   6.000000    3.000000
13                paymentservice2-0      trace.from_checkoutservice2-0.row_count    1.000000    4.000000   3.000000    3.000000
14               checkoutservice2-0         trace.to_paymentservice2-0.row_count    1.000000    4.000000   3.000000    3.000000
15               checkoutservice2-0  trace.to_productcatalogservice2-0.row_count    2.000000    8.000000   6.000000    3.000000
16                       frontend-2      trace.to_shippingservice-2.duration_p95    0.000066    0.000259   0.000193    2.924242
17                shippingservice-2           trace.from_frontend-2.duration_p95    0.000066    0.000259   0.000193    2.924242
18                       frontend-2      trace.to_checkoutservice-2.duration_p95    0.036576    0.098740   0.062164    1.699584
19                checkoutservice-2           trace.from_frontend-2.duration_p95    0.036576    0.098740   0.062164    1.699584, 'top10_components':                    cmdb_id  sum_abs_delta  num_traces_changed                 top_trace_by_abs_delta
0               frontend-0     287.025050                  88   trace.to_currencyservice-1.row_count
1               frontend-2     199.181368                  92                   trace.self.row_count
2               frontend-1     193.036279                  92                   trace.self.row_count
3              frontend2-0     157.037407                  36  trace.to_currencyservice2-0.row_count
4        checkoutservice-1      86.086158                  84                   trace.self.row_count
5  productcatalogservice-1      82.002989                  40        trace.from_frontend-0.row_count
6  productcatalogservice-0      82.002382                  40        trace.from_frontend-0.row_count
7       currencyservice2-0      81.004557                   8       trace.from_frontend2-0.row_count
8  productcatalogservice-2      80.001544                  40        trace.from_frontend-0.row_count
9       checkoutservice2-0      69.030307                  36                   trace.self.row_count}}```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz

# reuse timezone and trace_df if present in kernel
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

# Ensure dtypes
trace_df['timestamp'] = trace_df['timestamp'].astype(int)
trace_df['cmdb_id'] = trace_df['cmdb_id'].astype(str)
trace_df['trace_name'] = trace_df['trace_name'].astype(str)
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

target_minutes = ["2022-03-20 12:45:00+08:00", "2022-03-20 12:49:00+08:00"]

# helper function to analyze one minute
def analyze_trace_minute_health(minute_str):
    T = pd.Timestamp(minute_str, tz=tz)
    T_utc = int(T.tz_convert('UTC').timestamp())
    prev_utc = T_utc - 60

    # select only rows at T or T-1
    sel = trace_df[trace_df['timestamp'].isin([prev_utc, T_utc])].copy()

    # exclude frontend cmdb_ids (starts with 'frontend' or 'frontend2')
    mask_frontend = sel['cmdb_id'].str.startswith('frontend') | sel['cmdb_id'].str.startswith('frontend2')
    sel = sel[~mask_frontend]

    # restrict to trace_name categories of interest
    type_mask = (
        sel['trace_name'].str.contains('duration_p95', na=False) |
        sel['trace_name'].str.contains('duration_mean', na=False) |
        sel['trace_name'].str.contains('error_rate', na=False) |
        sel['trace_name'].str.contains('row_count', na=False)
    )
    sel = sel[type_mask].copy()

    # pivot to get prev and curr values per (cmdb_id, trace_name)
    if sel.empty:
        empty_cols = ['cmdb_id','trace_name','value_prev','value_curr','delta','rel_change']
        empty_df = pd.DataFrame(columns=empty_cols)
        empty_comp = pd.DataFrame(columns=['cmdb_id','sum_abs_delta','num_traces_changed','top_trace_by_abs_delta'])
        return {
            'A_top10_duration_error_by_abs_delta': empty_df,
            'B_top10_rowcount_negative_drops': empty_df,
            'C_top10_components_by_sum_abs_delta': empty_comp
        }

    pivot = sel.pivot_table(
        index=['cmdb_id','trace_name'],
        columns='timestamp',
        values='value',
        aggfunc='first'
    ).reset_index()

    col_map = {}
    if prev_utc in pivot.columns:
        col_map[prev_utc] = 'value_prev'
    if T_utc in pivot.columns:
        col_map[T_utc] = 'value_curr'
    pivot = pivot.rename(columns=col_map)

    if ('value_prev' not in pivot.columns) or ('value_curr' not in pivot.columns):
        empty_cols = ['cmdb_id','trace_name','value_prev','value_curr','delta','rel_change']
        empty_df = pd.DataFrame(columns=empty_cols)
        empty_comp = pd.DataFrame(columns=['cmdb_id','sum_abs_delta','num_traces_changed','top_trace_by_abs_delta'])
        return {
            'A_top10_duration_error_by_abs_delta': empty_df,
            'B_top10_rowcount_negative_drops': empty_df,
            'C_top10_components_by_sum_abs_delta': empty_comp
        }

    df_pairs = pivot.dropna(subset=['value_prev','value_curr']).copy()
    if df_pairs.empty:
        empty_cols = ['cmdb_id','trace_name','value_prev','value_curr','delta','rel_change']
        empty_df = pd.DataFrame(columns=empty_cols)
        empty_comp = pd.DataFrame(columns=['cmdb_id','sum_abs_delta','num_traces_changed','top_trace_by_abs_delta'])
        return {
            'A_top10_duration_error_by_abs_delta': empty_df,
            'B_top10_rowcount_negative_drops': empty_df,
            'C_top10_components_by_sum_abs_delta': empty_comp
        }

    # compute delta and rel_change
    df_pairs['delta'] = df_pairs['value_curr'] - df_pairs['value_prev']
    def rel_change_calc(row):
        prev = row['value_prev']
        d = row['delta']
        if pd.isna(prev) or prev == 0:
            return d
        return d / abs(prev)
    df_pairs['rel_change'] = df_pairs.apply(rel_change_calc, axis=1)
    df_pairs['abs_delta'] = df_pairs['delta'].abs()

    # A) Top 10 by absolute delta but only for duration_p95 OR duration_mean OR error_rate
    mask_A = (
        df_pairs['trace_name'].str.contains('duration_p95', na=False) |
        df_pairs['trace_name'].str.contains('duration_mean', na=False) |
        df_pairs['trace_name'].str.contains('error_rate', na=False)
    )
    A_df = df_pairs[mask_A].sort_values(by='abs_delta', ascending=False).head(10)[
        ['cmdb_id','trace_name','value_prev','value_curr','delta','rel_change']
    ].reset_index(drop=True)

    # B) Top 10 largest negative drops in row_count (delta < 0), ordered most negative first
    mask_B = df_pairs['trace_name'].str.contains('row_count', na=False) & (df_pairs['delta'] < 0)
    B_df = df_pairs[mask_B].sort_values(by='delta', ascending=True).head(10)[
        ['cmdb_id','trace_name','value_prev','value_curr','delta','rel_change']
    ].reset_index(drop=True)

    # C) Component-level aggregation across the selected trace metrics (all four types)
    sel_types_mask = (
        df_pairs['trace_name'].str.contains('duration_p95', na=False) |
        df_pairs['trace_name'].str.contains('duration_mean', na=False) |
        df_pairs['trace_name'].str.contains('error_rate', na=False) |
        df_pairs['trace_name'].str.contains('row_count', na=False)
    )
    comp_df = df_pairs[sel_types_mask].copy()
    if comp_df.empty:
        empty_comp = pd.DataFrame(columns=['cmdb_id','sum_abs_delta','num_traces_changed','top_trace_by_abs_delta'])
        return {
            'A_top10_duration_error_by_abs_delta': A_df,
            'B_top10_rowcount_negative_drops': B_df,
            'C_top10_components_by_sum_abs_delta': empty_comp
        }

    comp_agg = comp_df.groupby('cmdb_id', observed=True).agg(
        sum_abs_delta = ('abs_delta', 'sum'),
        num_traces_changed = ('trace_name', 'nunique')
    ).reset_index()

    # find top trace by abs_delta per component
    top_trace = comp_df.sort_values(by='abs_delta', ascending=False).groupby('cmdb_id', observed=True).first().reset_index()
    top_trace = top_trace[['cmdb_id','trace_name']].rename(columns={'trace_name':'top_trace_by_abs_delta'})

    comp_agg = comp_agg.merge(top_trace, on='cmdb_id', how='left')
    comp_top10 = comp_agg.sort_values(by='sum_abs_delta', ascending=False).head(10)[
        ['cmdb_id','sum_abs_delta','num_traces_changed','top_trace_by_abs_delta']
    ].reset_index(drop=True)

    return {
        'A_top10_duration_error_by_abs_delta': A_df,
        'B_top10_rowcount_negative_drops': B_df,
        'C_top10_components_by_sum_abs_delta': comp_top10
    }

# Run analysis for both target minutes and cache results
trace_minute_results = {}
for m in target_minutes:
    trace_minute_results[m] = analyze_trace_minute_health(m)

# Display the compact structured results (DataFrames are limited to requested maxima)
trace_minute_results

```
Out[9]:
```
```python
# Summarize the trace delta health analysis for the two target minutes using cached 'trace_minute_results'.
# The summary is plain English and compact.
try:
    trace_minute_results
except NameError:
    raise RuntimeError("Expected 'trace_minute_results' to exist in the kernel from prior step.")

m1 = "2022-03-20 12:45:00+08:00"
m2 = "2022-03-20 12:49:00+08:00"

r1 = trace_minute_results[m1]
r2 = trace_minute_results[m2]

summary_lines = [
    "Trace-based health snapshot (excluded frontends; metrics: duration_p95, duration_mean, error_rate, row_count).",
    "",
    f"Minute {m1}:",
    "- A) Largest absolute duration/error deltas (top examples):",
]
# Representative top A entries for m1
for row in r1['A_top10_duration_error_by_abs_delta'].head(3).itertuples(index=False):
    summary_lines.append(f"  * {row.cmdb_id} | {row.trace_name}: prev={row.value_prev} → curr={row.value_curr}, delta={row.delta}, rel_change={row.rel_change}")

summary_lines += [
    "- B) Largest drops in row_count (top examples):",
]
for row in r1['B_top10_rowcount_negative_drops'].head(3).itertuples(index=False):
    summary_lines.append(f"  * {row.cmdb_id} | {row.trace_name}: prev={row.value_prev} → curr={row.value_curr}, drop={row.delta} (rel={row.rel_change})")

summary_lines += [
    "- C) Top affected components by total change (top 5):",
]
for row in r1['C_top10_components_by_sum_abs_delta'].head(5).itertuples(index=False):
    summary_lines.append(f"  * {row.cmdb_id}: sum_abs_delta={row.sum_abs_delta}, traces_changed={int(row.num_traces_changed)}, top_trace={row.top_trace_by_abs_delta}")

summary_lines += [
    "",
    f"Minute {m2}:",
    "- A) Largest absolute duration/error deltas (top examples):",
]
for row in r2['A_top10_duration_error_by_abs_delta'].head(3).itertuples(index=False):
    summary_lines.append(f"  * {row.cmdb_id} | {row.trace_name}: prev={row.value_prev} → curr={row.value_curr}, delta={row.delta}, rel_change={row.rel_change}")

summary_lines += [
    "- B) Largest drops in row_count (top examples):",
]
for row in r2['B_top10_rowcount_negative_drops'].head(3).itertuples(index=False):
    summary_lines.append(f"  * {row.cmdb_id} | {row.trace_name}: prev={row.value_prev} → curr={row.value_curr}, drop={row.delta} (rel={row.rel_change})")

summary_lines += [
    "- C) Top affected components by total change (top 5):",
]
for row in r2['C_top10_components_by_sum_abs_delta'].head(5).itertuples(index=False):
    summary_lines.append(f"  * {row.cmdb_id}: sum_abs_delta={row.sum_abs_delta}, traces_changed={int(row.num_traces_changed)}, top_trace={row.top_trace_by_abs_delta}")

summary_lines += [
    "",
    "Key interpretation:",
    "- At 12:45 the biggest duration-related changes are in checkoutservice variants (several duration_p95/mean moves).",
    "- Also at 12:45 the largest drops in row_count affect productcatalogservice2 and currencyservice variants (notably productcatalogservice2: -70), indicating downstream request drop/decline after front-end errors.",
    "- Component totals at 12:45 show currencyservice and productcatalogservice family as principal contributors to total trace change.",
    "- At 12:49 there is a clear latency spike in checkoutservice-2 (duration_p95 increased ~0.0366→0.0987, large rel_change ≈ 1.7), alongside substantial row_count drops in currencyservice and productcatalogservice instances and reduced self.row_count in some checkout pods.",
    "- Overall, the signals point to downstream service instability around checkoutservice (latency increases) and productcatalogservice/currencyservice (reduced served requests) coinciding with frontend error-log peaks. These downstream services are high-priority candidates for root-cause investigation (check service logs, resource metrics, network/timeout traces).",
    "",
    "If you want the exact compact tables (A:10, B:10, C:10) returned as DataFrames for either minute, I can output them next."
]

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

{'2022-03-20 12:45:00+08:00': {'A_top10_duration_error_by_abs_delta': timestamp            cmdb_id                           trace_name  value_prev  value_curr     delta  rel_change
0          checkoutservice-1  trace.from_frontend-1.duration_mean    0.072437    0.035656 -0.036781   -0.507765
1          checkoutservice-1   trace.from_frontend-1.duration_p95    0.072437    0.040876 -0.031561   -0.435703
2          checkoutservice-2   trace.from_frontend-0.duration_p95    0.038023    0.029654 -0.008369   -0.220104
3          checkoutservice-2  trace.from_frontend-0.duration_mean    0.037269    0.029654 -0.007615   -0.204325
4          checkoutservice-1  trace.from_frontend-0.duration_mean    0.028682    0.036211  0.007529    0.262499
5          checkoutservice-1   trace.from_frontend-0.duration_p95    0.028921    0.036211  0.007290    0.252066
6          checkoutservice-0   trace.from_frontend-1.duration_p95    0.035179    0.039334  0.004155    0.118110
7          checkoutservice-2   trace.from_frontend-2.duration_p95    0.036682    0.040631  0.003949    0.107655
8          checkoutservice-0  trace.from_frontend-1.duration_mean    0.035179    0.038633  0.003454    0.098184
9          checkoutservice-1   trace.from_frontend-2.duration_p95    0.041016    0.038358 -0.002658   -0.064804, 'B_top10_rowcount_negative_drops': timestamp                   cmdb_id                                     trace_name  value_prev  value_curr  delta  rel_change
0          productcatalogservice2-0               trace.from_frontend2-0.row_count       341.0       271.0  -70.0   -0.205279
1                 currencyservice-0                trace.from_frontend-0.row_count       117.0       102.0  -15.0   -0.128205
2                 currencyservice-1                trace.from_frontend-0.row_count       117.0       102.0  -15.0   -0.128205
3                      adservice2-0               trace.from_frontend2-0.row_count        48.0        34.0  -14.0   -0.291667
4                 currencyservice-2                trace.from_frontend-0.row_count       116.0       102.0  -14.0   -0.120690
5                    cartservice2-0               trace.from_frontend2-0.row_count        65.0        52.0  -13.0   -0.200000
6          recommendationservice2-0                           trace.self.row_count        54.0        41.0  -13.0   -0.240741
7          recommendationservice2-0               trace.from_frontend2-0.row_count        54.0        41.0  -13.0   -0.240741
8                    cartservice2-0                           trace.self.row_count        72.0        65.0   -7.0   -0.097222
9           productcatalogservice-0  trace.from_recommendationservice2-0.row_count        18.0        13.0   -5.0   -0.277778, 'C_top10_components_by_sum_abs_delta':                     cmdb_id  sum_abs_delta  num_traces_changed            top_trace_by_abs_delta
0         currencyservice-1      81.000114                  24   trace.from_frontend-2.row_count
1         currencyservice-0      81.000066                  24   trace.from_frontend-2.row_count
2         currencyservice-2      78.000051                  24   trace.from_frontend-2.row_count
3   productcatalogservice-0      72.004475                  40   trace.from_frontend-0.row_count
4        currencyservice2-0      72.002554                   4  trace.from_frontend2-0.row_count
5   productcatalogservice-1      71.006886                  40   trace.from_frontend-0.row_count
6   productcatalogservice-2      71.003395                  40   trace.from_frontend-0.row_count
7  productcatalogservice2-0      70.000278                   4  trace.from_frontend2-0.row_count
8         checkoutservice-2      40.028043                  84              trace.self.row_count
9  recommendationservice2-0      39.000317                  20              trace.self.row_count}, '2022-03-20 12:49:00+08:00': {'A_top10_duration_error_by_abs_delta': timestamp             cmdb_id                           trace_name  value_prev  value_curr     delta  rel_change
0           checkoutservice-2   trace.from_frontend-2.duration_p95    0.036576    0.098740  0.062164    1.699584
1           checkoutservice-1   trace.from_frontend-2.duration_p95    0.082785    0.036068 -0.046717   -0.564317
2           checkoutservice-2  trace.from_frontend-2.duration_mean    0.032188    0.070333  0.038145    1.185069
3           checkoutservice-1  trace.from_frontend-2.duration_mean    0.061638    0.036068 -0.025570   -0.414841
4          checkoutservice2-0         trace.from_root.duration_p95    0.053449    0.066347  0.012898    0.241314
5          checkoutservice2-0  trace.from_frontend2-0.duration_p95    0.099570    0.107893  0.008323    0.083589
6           checkoutservice-2   trace.from_frontend-1.duration_p95    0.028818    0.036497  0.007679    0.266465
7           checkoutservice-2  trace.from_frontend-1.duration_mean    0.028818    0.036497  0.007679    0.266465
8           checkoutservice-0  trace.from_frontend-2.duration_mean    0.031645    0.036919  0.005274    0.166661
9           checkoutservice-1   trace.from_frontend-1.duration_p95    0.038923    0.033980 -0.004943   -0.126994, 'B_top10_rowcount_negative_drops': timestamp                   cmdb_id                        trace_name  value_prev  value_curr  delta  rel_change
0                 currencyservice-1   trace.from_frontend-0.row_count       145.0       101.0  -44.0   -0.303448
1                 currencyservice-2   trace.from_frontend-0.row_count       145.0       101.0  -44.0   -0.303448
2                 currencyservice-0   trace.from_frontend-0.row_count       145.0       101.0  -44.0   -0.303448
3          productcatalogservice2-0  trace.from_frontend2-0.row_count       331.0       293.0  -38.0   -0.114804
4                 checkoutservice-1              trace.self.row_count        56.0        24.0  -32.0   -0.571429
5                 checkoutservice-0              trace.self.row_count        47.0        24.0  -23.0   -0.489362
6           productcatalogservice-0   trace.from_frontend-1.row_count       155.0       133.0  -22.0   -0.141935
7           productcatalogservice-1   trace.from_frontend-1.row_count       155.0       133.0  -22.0   -0.141935
8           productcatalogservice-2   trace.from_frontend-1.row_count       155.0       134.0  -21.0   -0.135484
9           productcatalogservice-0   trace.from_frontend-2.row_count       208.0       189.0  -19.0   -0.091346, 'C_top10_components_by_sum_abs_delta':                    cmdb_id  sum_abs_delta  num_traces_changed            top_trace_by_abs_delta
0        checkoutservice-1      86.086158                  84              trace.self.row_count
1  productcatalogservice-1      82.002989                  40   trace.from_frontend-0.row_count
2  productcatalogservice-0      82.002382                  40   trace.from_frontend-0.row_count
3       currencyservice2-0      81.004557                   8  trace.from_frontend2-0.row_count
4  productcatalogservice-2      80.001544                  40   trace.from_frontend-0.row_count
5       checkoutservice2-0      69.030307                  36              trace.self.row_count
6        currencyservice-0      65.000071                  24   trace.from_frontend-0.row_count
7        checkoutservice-0      64.014768                  88              trace.self.row_count
8        currencyservice-1      62.000075                  24   trace.from_frontend-0.row_count
9        currencyservice-2      61.000083                  24   trace.from_frontend-0.row_count}}```
```

In [ ]:
# In[10]:

import pandas as pd
import pytz

# Reuse variables where possible
try:
    log_df
except NameError:
    log_df = pd.read_csv("log.csv")

tz = pytz.timezone('Asia/Shanghai')

# Ensure dtypes
log_df['timestamp'] = log_df['timestamp'].astype(int)
log_df['cmdb_id'] = log_df['cmdb_id'].astype(str)
if 'log_name' in log_df.columns:
    log_df['log_name'] = log_df['log_name'].astype(str)
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce').fillna(0)

# Incident window (inclusive) in Asia/Shanghai -> convert to UTC epoch seconds
start_ts = int(pd.Timestamp("2022-03-20T12:30:00+08:00", tz=tz).tz_convert('UTC').timestamp())
end_ts   = int(pd.Timestamp("2022-03-20T12:59:59+08:00", tz=tz).tz_convert('UTC').timestamp())

# Candidate services/pods list
candidates = [
    "checkoutservice", "checkoutservice-0", "checkoutservice-1", "checkoutservice-2", "checkoutservice2-0",
    "productcatalogservice", "productcatalogservice-0", "productcatalogservice-1", "productcatalogservice-2", "productcatalogservice2-0",
    "currencyservice", "currencyservice-0", "currencyservice-1", "currencyservice-2", "currencyservice2-0"
]

# 1) Filter to window and candidates
log_win = log_df[
    (log_df['timestamp'] >= start_ts) &
    (log_df['timestamp'] <= end_ts) &
    (log_df['cmdb_id'].isin(candidates))
].copy()

# Add Asia/Shanghai minute floor string
log_win['_dt_shanghai'] = pd.to_datetime(log_win['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
log_win['_minute_shanghai'] = log_win['_dt_shanghai'].dt.floor('min').astype(str)

# Aggregate per-minute per-cmdb_id per log_name
agg = (
    log_win
    .groupby(['_minute_shanghai', 'cmdb_id', 'log_name'], observed=True)['value']
    .sum()
    .reset_index()
)

# Pivot so we have error_count and row_count columns
pivot = agg.pivot_table(index=['_minute_shanghai','cmdb_id'], columns='log_name', values='value', aggfunc='first').reset_index()
# Ensure columns exist
if 'log.error_count' not in pivot.columns:
    pivot['log.error_count'] = 0
if 'log.row_count' not in pivot.columns:
    pivot['log.row_count'] = 0

# Normalize column names
pivot = pivot.rename(columns={'log.error_count':'error_count_sum','log.row_count':'row_count_sum'})

# Fill NaN with 0
pivot['error_count_sum'] = pivot['error_count_sum'].fillna(0).astype(int)
pivot['row_count_sum'] = pivot['row_count_sum'].fillna(0).astype(int)

# 1) Top 20 (minute, cmdb_id) rows ranked by error_count_sum
top20_rows = pivot.sort_values(by='error_count_sum', ascending=False).head(20)[
    ['_minute_shanghai','cmdb_id','error_count_sum','row_count_sum']
].reset_index(drop=True)

# 2) For the two target minutes return per-cmdb_id summaries for candidate list
target_minutes = ["2022-03-20 12:45:00+08:00", "2022-03-20 12:49:00+08:00"]
minute_summaries = {}
for minute in target_minutes:
    df_min = pivot[pivot['_minute_shanghai'] == minute][['cmdb_id','error_count_sum','row_count_sum']].copy()
    # ensure all candidates present (with zeros if missing)
    df_min = df_min.set_index('cmdb_id').reindex(candidates).fillna(0).reset_index()
    df_min['error_count_sum'] = df_min['error_count_sum'].astype(int)
    df_min['row_count_sum'] = df_min['row_count_sum'].astype(int)
    minute_summaries[minute] = df_min

# 3) Overall (window) per-cmdb_id totals for these candidate services
overall = (
    pivot
    .groupby('cmdb_id', observed=True)[['error_count_sum','row_count_sum']]
    .sum()
    .reset_index()
    .rename(columns={'error_count_sum':'total_error_count','row_count_sum':'total_row_count'})
)

overall_top10 = overall.sort_values(by='total_error_count', ascending=False).head(10).reset_index(drop=True)

# Compact outputs (DataFrames limited as requested)
top20_rows, minute_summaries, overall_top10

```
Out[10]:
```
```python
# Summarize log.csv aggregates for candidate services in the incident window (plain English).
# Reuse cached DataFrames: top20_rows, minute_summaries, overall_top10 from prior execution.

try:
    top20_rows
    minute_summaries
    overall_top10
except NameError:
    raise RuntimeError("Expected aggregated variables to exist in the kernel.")

summary_lines = [
    "Log-based summary for candidate services (window: 2022-03-20 12:30:00+08 → 12:59:59+08):",
    "",
    "1) Top 20 (minute, cmdb_id) by log.error_count:",
    "- Observation: For all candidate services/pods in the window, log.error_count sums are zero. The top-20 ranking therefore shows rows with error_count_sum = 0 and is effectively ordered by presence/row_count.",
    "- Examples (minute, cmdb_id, error_count_sum, row_count_sum):",
    f"  * 2022-03-20 12:59:00+08:00 | productcatalogservice-2 — errors: 0, rows: 1110",
    f"  * 2022-03-20 12:30:00+08:00 | checkoutservice-0 — errors: 0, rows: 79",
    f"  * 2022-03-20 12:30:00+08:00 | checkoutservice-1 — errors: 0, rows: 85",
    f"  * 2022-03-20 12:30:00+08:00 | checkoutservice-2 — errors: 0, rows: 85",
    f"  * 2022-03-20 12:30:00+08:00 | currencyservice-0 — errors: 0, rows: 904",
    "",
    "2) Per-cmdb_id summaries for the two target minutes (12:45 and 12:49):",
    "- Key point: All candidate cmdb_ids show error_count_sum = 0 at both minutes.",
    "- 2022-03-20 12:45:00+08: (cmdb_id → error_count_sum, row_count_sum examples):",
    "  * productcatalogservice-0: errors=0, rows=1205",
    "  * productcatalogservice-1: errors=0, rows=1229",
    "  * productcatalogservice-2: errors=0, rows=1214",
    "  * currencyservice-0: errors=0, rows=1000; currencyservice-1: errors=0, rows=998; currencyservice-2: errors=0, rows=1007",
    "  * checkoutservice-0/1/2: errors=0, rows ≈ 71 / 68 / 82 (respectively)",
    "",
    "- 2022-03-20 12:49:00+08: (cmdb_id → error_count_sum, row_count_sum examples):",
    "  * productcatalogservice-0: errors=0, rows=1230",
    "  * productcatalogservice-1: errors=0, rows=1203",
    "  * productcatalogservice-2: errors=0, rows=1183",
    "  * currencyservice-0/1/2: errors=0, rows ≈ 905 / 905 / 916",
    "  * checkoutservice-0/1/2: errors=0, rows ≈ 51 / 54 / 67",
    "",
    "3) Overall window totals (candidate services) — top 10 by total_error_count (note: all zeros):",
    "- All total_error_count = 0 across these candidates. Use row counts to see traffic volume instead.",
    "- Top by total_row_count (examples):",
    f"  * productcatalogservice-2: total_row_count = {int(overall_top10.loc[2,'total_row_count']) if ('total_row_count' in overall_top10.columns and overall_top10.shape[0]>2) else overall_top10.loc[0,'total_row_count']}",
    f"  * productcatalogservice-1: total_row_count = {int(overall_top10.loc[1,'total_row_count']) if overall_top10.shape[0]>1 else overall_top10.loc[0,'total_row_count']}",
    f"  * productcatalogservice-0: total_row_count = {int(overall_top10.loc[0,'total_row_count'])}",
    f"  * currencyservice-2/1/0: total_row_count ≈ {int(overall_top10.loc[5,'total_row_count'])} / {int(overall_top10.loc[4,'total_row_count'])} / {int(overall_top10.loc[3,'total_row_count'])}",
    f"  * checkoutservice-0/1/2: total_row_count ≈ {int(overall_top10.loc[0,'total_row_count']) if overall_top10.shape[0]>0 else 0} (these three are ~2.2k each in the window)",
    "",
    "Interpretation and next steps:",
    "- No backend error_count log spikes are observed for these candidate services during the incident window; log.error_count aggregates are all zero.",
    "- Productcatalogservice and currencyservice handled the largest request volumes (high row_count totals), while checkoutservice pods show comparatively low per-minute row counts.",
    "- Because error_count is zero, frontend error warnings observed earlier are not mirrored by backend log.error_count increments for these services. However, trace and metric analyses showed row_count drops and latency spikes for downstream services (productcatalogservice, currencyservice, checkoutservice variants).",
    "- Recommendation: investigate traces, application logs (INFO/WARN/ERROR text) and metrics for productcatalogservice and currencyservice (and checkoutservice variants) rather than relying solely on log.error_count; also inspect network/timeouts and resource metrics around the two peak minutes.",
    "",
    "If you want, I can return the compact DataFrame tables (top20 minute/cmdb rows, per-minute candidate summaries, and overall totals) as CSV-like prints limited to the maxima."
]

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(log_name           _minute_shanghai                  cmdb_id  error_count_sum  row_count_sum
0         2022-03-20 12:59:00+08:00  productcatalogservice-2                0           1110
1         2022-03-20 12:30:00+08:00        checkoutservice-0                0             79
2         2022-03-20 12:30:00+08:00        checkoutservice-1                0             85
3         2022-03-20 12:30:00+08:00        checkoutservice-2                0             85
4         2022-03-20 12:30:00+08:00        currencyservice-0                0            904
5         2022-03-20 12:30:00+08:00        currencyservice-1                0            905
6         2022-03-20 12:30:00+08:00        currencyservice-2                0            918
7         2022-03-20 12:30:00+08:00  productcatalogservice-0                0            989
8         2022-03-20 12:30:00+08:00  productcatalogservice-1                0           1163
9         2022-03-20 12:30:00+08:00  productcatalogservice-2                0           1169
10        2022-03-20 12:31:00+08:00        checkoutservice-0                0             71
11        2022-03-20 12:31:00+08:00        checkoutservice-1                0             51
12        2022-03-20 12:31:00+08:00        checkoutservice-2                0             74
13        2022-03-20 12:31:00+08:00        currencyservice-0                0            900
14        2022-03-20 12:55:00+08:00        currencyservice-2                0            888
15        2022-03-20 12:55:00+08:00  productcatalogservice-0                0           1196
16        2022-03-20 12:55:00+08:00  productcatalogservice-1                0           1193
17        2022-03-20 12:55:00+08:00  productcatalogservice-2                0           1181
18        2022-03-20 12:56:00+08:00        checkoutservice-0                0             54
19        2022-03-20 12:56:00+08:00        checkoutservice-1                0             71, {'2022-03-20 12:45:00+08:00': log_name                   cmdb_id  error_count_sum  row_count_sum
0                  checkoutservice                0              0
1                checkoutservice-0                0             71
2                checkoutservice-1                0             68
3                checkoutservice-2                0             82
4               checkoutservice2-0                0              0
5            productcatalogservice                0              0
6          productcatalogservice-0                0           1205
7          productcatalogservice-1                0           1229
8          productcatalogservice-2                0           1214
9         productcatalogservice2-0                0              0
10                 currencyservice                0              0
11               currencyservice-0                0           1000
12               currencyservice-1                0            998
13               currencyservice-2                0           1007
14              currencyservice2-0                0              0, '2022-03-20 12:49:00+08:00': log_name                   cmdb_id  error_count_sum  row_count_sum
0                  checkoutservice                0              0
1                checkoutservice-0                0             51
2                checkoutservice-1                0             54
3                checkoutservice-2                0             67
4               checkoutservice2-0                0              0
5            productcatalogservice                0              0
6          productcatalogservice-0                0           1230
7          productcatalogservice-1                0           1203
8          productcatalogservice-2                0           1183
9         productcatalogservice2-0                0              0
10                 currencyservice                0              0
11               currencyservice-0                0            905
12               currencyservice-1                0            905
13               currencyservice-2                0            916
14              currencyservice2-0                0              0}, log_name                  cmdb_id  total_error_count  total_row_count
0               checkoutservice-0                  0             2231
1               checkoutservice-1                  0             2223
2               checkoutservice-2                  0             2216
3               currencyservice-0                  0            27683
4               currencyservice-1                  0            27687
5               currencyservice-2                  0            27734
6         productcatalogservice-0                  0            35565
7         productcatalogservice-1                  0            35670
8         productcatalogservice-2                  0            35717)```
```